# Connecting Online Opinions with Urban Statistics: WWW '27 Analysis

Starting from two archived source inputs, this notebook executes:
- Preprocessing and exact parity checks against archived reference tables;
- Reproduction of all baseline reference models;
- All three measurement, association, and robustness algorithms (Algorithms 1–3).

**Running Locally**
1. Open this notebook and select a Python 3.12 kernel with `numpy`, `pandas`, `scipy`, and `statsmodels` (see `requirements.txt`).
2. Run all cells (`~2 min` total; bootstrap calibration takes `~80 s`; set `QUICK = True` for a `~20 s` smoke check).
3. Cell 1 automatically locates the repository root (`PROJECT`) by walking upward from the working directory and sets the working folder to `data/`.

**Sections**

| Section | Contents | Uses real outcomes |
|---|---|---|
| 1 | Preprocessing: rebuild 38-city analytic table from census CSV and archived city-keyword dictionary | No |
| 2 | Baseline reproduction: verify parity with archived reference outputs | Yes (published baseline) |
| 3 | Algorithm 1: decompose domain scores into tone, composition, and coverage; reliability; REML variance | No |
| 4 | Design diagnostics: coverage vs. corpus size, network connectivity under removal, permutation size, bootstrap calibration | No |
| 5 | Zero-effect retention simulation (Algorithm 3, lines 9–11) | No |
| 6 | Keyword screen (Algorithm 3, line 8): coding template, inter-coder κ, 30% cap & connectivity checks | No |
| 7 | Association, outcome-side/score-side decompositions, RQ2 partial correlations, and robustness (Algorithm 2; Algorithm 3, lines 1–8) | `confirmatory` mode only |
| 8 | Run log and output inventory | — |

**Execution Modes**
- `MODE = 'confirmatory'` (default once the keyword screen is frozen): runs on real unshuffled urban statistics and writes to a timestamped `results/www_short/confirmatory_<timestamp>/` folder without overwriting existing `results/www_short/confirmatory/` files.
- `MODE = 'dry_run'`: permutes the entire administrative block (outcomes, controls, and urban indicators) jointly across cities while holding keyword signals fixed, verifying code execution without generating real associations in `results/www_short/dry_run/`.

In [1]:
# ---- 0. Settings ----------------------------------------------------------------------------
from pathlib import Path
import os

def find_repo_root() -> Path:
    cwd = Path.cwd().resolve()
    for base in (cwd, *cwd.parents):
        if (base / 'data' / 'census_cities.csv').exists() and (base / 'manifest_sha256.json').exists():
            return base
    raise FileNotFoundError(f'Could not locate repository root from {cwd}')

PROJECT = find_repo_root()
DATA_DIR = PROJECT / 'data'
os.chdir(DATA_DIR)                        # working directory = the data folder
OUT = PROJECT / 'results' / 'www_short'   # everything this notebook writes goes here

MODE = 'confirmatory'          # 'dry_run': census block shuffled across cities | 'confirmatory': real data, after the screen is frozen
SEED = 20261104
QUICK = False             # True: fewer resamples and simulations, for a fast smoke test
RUN_DESIGN_DIAGNOSTICS = True

# Fixed settings from the algorithm document
B_BOOT = 200 if QUICK else 2000           # city bootstrap resamples; the interval is descriptive only
R_NULL = 50 if QUICK else 500             # null-simulation runs
RETENTION_THRESHOLD = 10                  # a keyword mean is archived only with more than 10 valid scores
THETA_GRID_STEP = 0.25 if QUICK else 0.05 # grid for the retention elasticity theta
SCREEN_MAX_DROP = 0.30                    # at most 30% of a domain's listed keywords may be screened out
RE_LRT_ALPHA = 0.20                       # random-effects tone used only where the boundary-corrected REML likelihood-ratio
                                          # test of city variance = 0 gives p below this (see 3.4)
SCREEN_CODES = DATA_DIR / 'keyword_screen_codes.csv'   # both coders' decisions, to be supplied
SCREEN_RULES = DATA_DIR / 'keyword_screen_rules.md'    # the written screening rules, to be supplied
SCREEN_FROZEN = True                      # frozen 2026-11-06: rules and final codes in data/, hashes in each run log
MAIN_RUN_BEFORE_SCREEN = False            # with MODE = 'confirmatory': run the main analysis on real data before the screen
                                          # is frozen; the screened rerun (7.8) then waits for the frozen screen

print('working directory:', Path.cwd().relative_to(PROJECT))
print('mode:', MODE, '| quick:', QUICK)

working directory: data
mode: confirmatory | quick: False


In [2]:
import ast, datetime, hashlib, json, platform, time, warnings
import numpy as np
import pandas as pd
import scipy
from scipy import optimize, stats
from scipy.linalg import cho_factor, cho_solve
import statsmodels
import statsmodels.api as sm
from statsmodels.stats.multitest import multipletests
from IPython.display import display

pd.set_option('display.width', 220)
pd.set_option('display.max_columns', 40)
pd.set_option('display.max_colwidth', 60)
rng = np.random.default_rng(SEED)
T_START = time.time()
for sub in ['preprocessing', 'original', 'measurement', 'diagnostics', 'keyword_screen']:
    (OUT / sub).mkdir(parents=True, exist_ok=True)


def sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


def save(frame, folder, name, index=False):
    path = Path(folder) / name if Path(folder).is_absolute() else OUT / folder / name
    frame.to_csv(path, index=index, float_format='%.12g')


CHECKS = []   # reproduction and identity checks, collected for the run log


def check(name, ok, detail=''):
    CHECKS.append({'check': name, 'passed': bool(ok), 'detail': str(detail)})
    print(('PASS  ' if ok else 'FAIL  ') + name + (f'   [{detail}]' if detail != '' else ''))
    if not ok:
        raise AssertionError(name)


def zscore(frame):
    """Population-SD standardization (ddof=0), as in the original analysis."""
    scales = frame.std(ddof=0)
    if np.any(np.asarray(scales) == 0):
        raise ValueError('A constant variable cannot be standardized')
    return (frame - frame.mean()) / scales


def fit_model(x_raw, y_raw):
    """Original model: OLS on standardized variables with intercept; HC3 with Student t on residual df."""
    if x_raw.isna().any().any() or y_raw.isna().any():
        raise ValueError('Missing values must be handled explicitly before fitting')
    x = sm.add_constant(zscore(x_raw), has_constant='add')
    y = zscore(y_raw)
    model = sm.OLS(y, x).fit()
    robust = model.get_robustcov_results(cov_type='HC3', use_t=True)
    return model, robust, x, y


def coefficient_rows(model, robust, branch, outcome, family, **meta):
    ci, rci = np.asarray(model.conf_int(alpha=0.05)), np.asarray(robust.conf_int(alpha=0.05))
    rows = []
    for i, term in enumerate(model.params.index):
        rows.append({'branch': branch, 'outcome': outcome, 'term': term, 'family': family,
                     'n': int(model.nobs), 'predictors': int(model.df_model), 'df_resid': int(model.df_resid),
                     'beta': float(model.params.iloc[i]), 'ols_se': float(model.bse.iloc[i]),
                     'ols_t': float(model.tvalues.iloc[i]), 'ols_p': float(model.pvalues.iloc[i]),
                     'ols_ci_low': float(ci[i, 0]), 'ols_ci_high': float(ci[i, 1]),
                     'hc3_se': float(robust.bse[i]), 'hc3_t': float(robust.tvalues[i]), 'hc3_p': float(robust.pvalues[i]),
                     'hc3_ci_low': float(rci[i, 0]), 'hc3_ci_high': float(rci[i, 1]),
                     'r_squared': float(model.rsquared), 'adjusted_r_squared': float(model.rsquared_adj),
                     'uncertainty_distribution': 'Student t with model residual df', **meta})
    return rows


def add_fdr(frame, family_columns, eligible=None):
    """Benjamini-Hochberg q within each named family; intercepts excluded unless `eligible` says otherwise."""
    out = frame.copy()
    out['ols_q_bh'] = np.nan
    out['hc3_q_bh'] = np.nan
    out['fdr_family_size'] = np.nan
    if eligible is None:
        eligible = out.term.ne('const')
    for _, subset in out.loc[eligible].groupby(family_columns, sort=False):
        for source in ('ols', 'hc3'):
            out.loc[subset.index, f'{source}_q_bh'] = multipletests(subset[f'{source}_p'].to_numpy(), method='fdr_bh')[1]
        out.loc[subset.index, 'fdr_family_size'] = len(subset)
    return out


def compare_to_archive(new, archived_path, keys, cols, rtol=1e-8, atol=1e-9):
    """Compare a recomputed table with an archived CSV on the given key columns."""
    old = pd.read_csv(archived_path)
    merged = old.merge(new, on=keys, suffixes=('_old', '_new'), how='outer', indicator=True)
    both = merged[merged['_merge'] == 'both']
    unmatched = int((merged['_merge'] != 'both').sum())
    diffs = [np.abs(both[f'{c}_old'].to_numpy(float) - both[f'{c}_new'].to_numpy(float)) for c in cols]
    worst = float(max(np.nanmax(d) if np.isfinite(d).any() else 0.0 for d in diffs))
    ok = unmatched == 0 and all(np.allclose(both[f'{c}_old'].to_numpy(float), both[f'{c}_new'].to_numpy(float),
                                            rtol=rtol, atol=atol, equal_nan=True) for c in cols)
    return ok, f'{len(both)} rows, {unmatched} unmatched, max |diff| {worst:.1e}'

## 1. Preprocessing

The pipeline starts from two archived source inputs:
- `census_cities.csv`: 49 U.S. metropolitan areas across 27 urban/census indicators;
- `city_aspects.npy`: city-to-keyword sentiment dictionary for 40 U.S. cities (`city -> {keyword: mean archived score}`), where a term was archived only when a city exceeded 10 scored mentions.

We follow the canonical baseline preprocessing and add three fields needed for the measurement decomposition:
- corpus-size proxy ($s_i$),
- city-by-keyword long table,
- basic keyword coverage summaries.

Each intermediate table is cross-checked against the archived baseline CSVs.

### 1.1 Input Integrity Check

Verify SHA-256 digests of the four canonical `data/` inputs against `manifest_sha256.json`.

In [3]:
INPUTS = ['census_cities.csv', 'city_aspects.npy', 'aspect_domain_dictionary.csv', 'aspect_keyword_means_long.csv']
manifest = json.loads((PROJECT / 'manifest_sha256.json').read_text())
INPUT_HASHES = {name: sha256(name) for name in INPUTS}
for name, digest in INPUT_HASHES.items():
    check(f'unchanged since packaging: {name}', manifest.get(f'data/{name}') == digest, digest[:12])

PASS  unchanged since packaging: census_cities.csv   [947b95ed3b18]
PASS  unchanged since packaging: city_aspects.npy   [c94e1d3578e7]
PASS  unchanged since packaging: aspect_domain_dictionary.csv   [3cca63ed5a3d]
PASS  unchanged since packaging: aspect_keyword_means_long.csv   [b354bad48723]


### 1.2 Expanding the City-Keyword Dictionary to Long Form

`city_aspects.npy` is a project-archived dictionary (`allow_pickle=True`). Flattening it yields `(city, keyword, score)` rows matching `aspect_keyword_means_long.csv`.

In [4]:
census = pd.read_csv('census_cities.csv')
city_aspects = np.load('city_aspects.npy', allow_pickle=True).item()   # project-owned archive only
print('census:', census.shape, '| cities in the keyword dictionary:', len(city_aspects))

long_all = pd.DataFrame([{'city': c, 'keyword': k, 'archived_keyword_mean': float(v)}
                         for c, kw in city_aspects.items() for k, v in kw.items()])
archived_long = pd.read_csv('aspect_keyword_means_long.csv')
same = (long_all.shape == archived_long.shape
        and (long_all[['city', 'keyword']].to_numpy() == archived_long[['city', 'keyword']].to_numpy()).all()
        and np.allclose(long_all.archived_keyword_mean, archived_long.archived_keyword_mean, rtol=0, atol=1e-13))
check('long table rebuilt from the NPY equals aspect_keyword_means_long.csv', same, f'{len(long_all)} city-term rows')
check('terms are lower case without padding', long_all.keyword.eq(long_all.keyword.str.lower().str.strip()).all())
check('no missing keyword means', long_all.archived_keyword_mean.notna().all())

census: (49, 27) | cities in the keyword dictionary: 40
PASS  long table rebuilt from the NPY equals aspect_keyword_means_long.csv   [18406 city-term rows]
PASS  terms are lower case without padding
PASS  no missing keyword means


### 1.3 Twelve Domains and 145 Keywords

Parse literal model definitions (`PAPER_GROUPS`, `TABLE3_PREDICTORS`, `EXCLUDED_METROS`) safely via `ast.literal_eval` from the archived baseline reproduction scripts without executing them.

In [5]:
def archived_constant(filename, name):
    """Read a literal model definition from an archived script without executing it."""
    tree = ast.parse((PROJECT / 'code' / filename).read_text(encoding='utf-8'))
    for st in tree.body:
        if isinstance(st, ast.Assign) and any(isinstance(t, ast.Name) and t.id == name for t in st.targets):
            return ast.literal_eval(st.value)
    raise KeyError((filename, name))


PAPER_GROUPS = archived_constant('table4_online_regression_reproduction.py', 'PAPER_GROUPS')
TABLE3_PREDICTORS = list(archived_constant('table3_objective_regression_reproduction.py', 'TABLE3_PREDICTORS'))
EXCLUDED_METROS = set(archived_constant('table3_objective_regression_reproduction.py', 'EXCLUDED_METROS'))
DOMAINS = list(PAPER_GROUPS)
dictionary = pd.DataFrame([{'domain': g, 'keyword': k} for g, ks in PAPER_GROUPS.items() for k in ks])
check('145 listed keywords in 12 domains, none listed twice',
      len(dictionary) == 145 and dictionary.keyword.is_unique and len(DOMAINS) == 12)
check('dictionary equals aspect_domain_dictionary.csv', dictionary.equals(pd.read_csv('aspect_domain_dictionary.csv')))
display(dictionary.groupby('domain', sort=False).size().rename('listed keywords').to_frame().T)

PASS  145 listed keywords in 12 domains, none listed twice
PASS  dictionary equals aspect_domain_dictionary.csv


domain,Public Transportation,Driving and Traffic,Economy and Income,Education,Culture and Art,Race/Immigration Discourse,Recreation,Social Problem,Governing,Policing,Neighborhood,City Development
listed keywords,22,11,13,5,7,13,14,20,15,8,5,12


### 1.4 City–Metro Crosswalk

Following the baseline rule:
- match the first city name in each MSA label to its city subreddit key;
- map the two aliases (`NYC`, `Salt Lake`) explicitly;
- exclude Washington and Birmingham (and the 9 metros outside the 40-city Reddit archive), leaving 38 cities.

In [6]:
SHARES = ['%Non Hispanic Black', '%Hispanic', '%Asian']
PAIRWISE = ['White-Black dissimilarity index', 'White-Hispanic dissimilarity index', 'White-Asian dissimilarity index']
CONTROLS = ['urban_pop_den_sq mi', 'median household income']
BEHAVIORS = ['commute_%drive', 'commute_%carpool', '%home ownership']
ALIASES = {'NYC': 'New York', 'Salt Lake': 'Salt Lake City'}
EXCLUDED_ASPECT_CITIES = {'Washington', 'Birmingham'}

short_names = census.Metro.str.split(r'-|,|/', n=1, regex=True).str[0]
check('metro first names are unique', not short_names.duplicated().any())
short_lookup = dict(zip(short_names, census.index))
rows = []
for city in city_aspects:
    name = ALIASES.get(city, city)
    if name not in short_lookup:
        raise ValueError(f'Unmapped archived city: {city}')
    r = int(short_lookup[name])
    included = city not in EXCLUDED_ASPECT_CITIES
    rows.append({'city': city, 'census_row': r, 'metro': census.loc[r, 'Metro'], 'published_sample': included,
                 'mapping': 'explicit alias' if city in ALIASES else 'archived metro first-name rule',
                 'exclusion_reason': '' if included else 'excluded by archived published-model code; upstream rationale not reconstructed'})
crosswalk = pd.DataFrame(rows)
CITIES = crosswalk.loc[crosswalk.published_sample, 'city'].tolist()
CITY_IDX = pd.Index(CITIES, name='city')
aligned = census.iloc[crosswalk.set_index('city').loc[CITIES, 'census_row'].to_numpy()].copy()
aligned.index = CITY_IDX
check('38 linked cities, one metro each', len(CITIES) == 38 and aligned.Metro.is_unique)
check('same metros as the archived objective model',
      set(aligned.Metro) == set(census.loc[~census.Metro.isin(EXCLUDED_METROS), 'Metro']))
old_cw = pd.read_csv(PROJECT / 'results' / 'city_metro_crosswalk.csv').fillna('')
check('crosswalk equals results/city_metro_crosswalk.csv',
      (crosswalk.fillna('').astype(str).to_numpy() == old_cw.astype(str).to_numpy()).all())

PASS  metro first names are unique
PASS  38 linked cities, one metro each
PASS  same metros as the archived objective model
PASS  crosswalk equals results/city_metro_crosswalk.csv


### 1.5 Domain Score $a_{ig}$, Coverage $n_{ig}$, and Missingness

- **Domain score ($a_{ig}$):** equal-weight mean across retained keywords in domain $g$ for city $i$; missing when $n_{ig} = 0$ (zero-filled in the archived reference model).
- **Coverage ($n_{ig}$):** count of retained keywords in `(city i, domain g)`.

In [7]:
score_rows, cover_rows = [], []
for city in CITIES:
    kw = city_aspects[city]
    row = {'city': city}
    for g, terms in PAPER_GROUPS.items():
        present = [k for k in terms if k in kw]
        row[g] = float(np.mean([float(kw[k]) for k in present])) if present else np.nan
        cover_rows.append({'city': city, 'domain': g, 'listed_keywords': len(terms), 'retained_keywords': len(present),
                           'score': row[g], 'missing': not present, 'retained_keyword_list': '|'.join(present)})
    score_rows.append(row)
A_SCORE = pd.DataFrame(score_rows).set_index('city')[DOMAINS]
coverage = pd.DataFrame(cover_rows)
N_COV = coverage.pivot(index='city', columns='domain', values='retained_keywords').loc[CITIES, DOMAINS]
missing = A_SCORE.isna()
check('27 missing city-domain cells in 16 cities', int(missing.sum().sum()) == 27 and int(missing.any(axis=1).sum()) == 16)
check('one genuinely observed zero score', int(A_SCORE.eq(0).sum().sum()) == 1)
old_scores = pd.read_csv(PROJECT / 'results' / 'aspect_groups_true_missing.csv', index_col='city').loc[CITIES, DOMAINS]
check('domain scores equal results/aspect_groups_true_missing.csv',
      np.allclose(A_SCORE, old_scores, rtol=0, atol=1e-10, equal_nan=True))
old_cov = pd.read_csv(PROJECT / 'results' / 'keyword_coverage_and_missingness.csv')
old_cov = old_cov[old_cov.city.isin(CITIES)].pivot(index='city', columns='group', values='available_keywords').loc[CITIES, DOMAINS]
check('coverage counts equal the archived coverage file', (old_cov.to_numpy() == N_COV.to_numpy()).all())
display(pd.DataFrame({'available cities': A_SCORE.notna().sum(),
                      'retained keywords, min–max': N_COV.min().astype(str) + '–' + N_COV.max().astype(str)}).T)

PASS  27 missing city-domain cells in 16 cities
PASS  one genuinely observed zero score
PASS  domain scores equal results/aspect_groups_true_missing.csv
PASS  coverage counts equal the archived coverage file


,Public Transportation,Driving and Traffic,Economy and Income,Education,Culture and Art,Race/Immigration Discourse,Recreation,Social Problem,Governing,Policing,Neighborhood,City Development
available cities,37,38,37,38,37,23,38,35,34,36,38,38
"retained keywords, min–max",0–15,2–8,0–11,1–4,0–5,0–11,1–9,0–15,0–11,0–7,1–5,3–12


### 1.6 Outcome Variables

Let $p_g$ denote the three minority population shares and $D_{gW}$ denote each White–minority dissimilarity index.

| Outcome | Definition |
|---|---|
| Archived composite | $C = \sum_g p_g D_{gW}$ |
| Combined group share | $S = \sum_g p_g$ |
| Normalized composite | $N = C / S$ |
| Equal-weight mean | $\bar D = \frac{1}{3}(D_{BW} + D_{HW} + D_{AW})$ |
| Pairwise indices | $D_{BW}, D_{HW}, D_{AW}$ |

Six outcomes are modeled in RQ1: $C, N, \bar D, D_{BW}, D_{HW}, D_{AW}$.

In [8]:
OUTCOME_LABELS = {'archived_composite': 'C', 'normalized_composite': 'N = C/S', 'equal_pairwise_mean': 'D-bar',
                  'd_black_white': 'D_BW', 'd_hispanic_white': 'D_HW', 'd_asian_white': 'D_AW',
                  'combined_group_share': 'S'}
OUTCOMES = list(OUTCOME_LABELS)[:-1]
outc = pd.DataFrame(index=CITY_IDX)
outc['archived_composite'] = (aligned[SHARES].to_numpy() * aligned[PAIRWISE].to_numpy()).sum(axis=1)
outc['combined_group_share'] = aligned[SHARES].sum(axis=1)
outc['normalized_composite'] = outc.archived_composite / outc.combined_group_share
outc['equal_pairwise_mean'] = aligned[PAIRWISE].mean(axis=1)
for key, src in zip(['d_black_white', 'd_hispanic_white', 'd_asian_white'], PAIRWISE):
    outc[key] = aligned[src]
OUT_Y = outc[list(OUTCOME_LABELS)]
old_out = pd.read_csv(PROJECT / 'results' / 'outcomes_38_metros.csv', index_col='city').loc[CITIES, list(OUTCOME_LABELS)]
check('outcomes equal results/outcomes_38_metros.csv', np.allclose(OUT_Y, old_out, rtol=1e-10, atol=1e-9))
r_cs = OUT_Y.archived_composite.corr(OUT_Y.combined_group_share)
check('r(C, S) = .927', round(r_cs, 3) == 0.927, f'{r_cs:.6f}')

PASS  outcomes equal results/outcomes_38_metros.csv
PASS  r(C, S) = .927   [0.926526]


### 1.7 Corpus-Size Proxy ($s_i$)

Raw comment counts were not archived alongside `city_aspects.npy`. Because keywords entered a city dictionary only after passing the 10-mention threshold, the total number of retained entries in a city's dictionary serves as a corpus-size proxy $s_i$.

Removing all 145 scored study keywords and recounting leaves a nearly identical proxy ($r = 1.000$), confirming that $s_i$ is not mechanically driven by the study keywords.

In [9]:
SIZE = pd.DataFrame({'terms_in_dictionary': long_all.groupby('city').size().reindex(CITY_IDX)})
SIZE['log_size'] = np.log(SIZE.terms_in_dictionary)
SIZE['log_pop'] = np.log(aligned[' pop_2010'])          # the census field name starts with a space
non_kw = long_all[~long_all.keyword.isin(dictionary.keyword)].groupby('city').size().reindex(CITY_IDX)
print(f"s_i = terms in the city's archived dictionary: min {SIZE.terms_in_dictionary.min()}, "
      f"median {SIZE.terms_in_dictionary.median():.0f}, max {SIZE.terms_in_dictionary.max()}")
print(f'r(log s, log pop) = {SIZE.log_size.corr(SIZE.log_pop):.2f} | '
      f'r(log s, log s without the 145 keywords) = {np.corrcoef(SIZE.log_size, np.log(non_kw))[0, 1]:.3f}')
check('corpus-size proxy ranges from 121 to 914', SIZE.terms_in_dictionary.min() == 121 and SIZE.terms_in_dictionary.max() == 914)

s_i = terms in the city's archived dictionary: min 121, median 381, max 914
r(log s, log pop) = 0.46 | r(log s, log s without the 145 keywords) = 1.000
PASS  corpus-size proxy ranges from 121 to 914


### 1.8 City–Keyword Long Table

Input to Algorithm 1: all retained `(city, domain, keyword, score)` cells across the 38 sample cities and 145 listed keywords.

In [10]:
cells = long_all.merge(dictionary, on='keyword')
cells = cells[cells.city.isin(CITIES)].rename(columns={'archived_keyword_mean': 'm'})[['city', 'domain', 'keyword', 'm']]
cells = cells.reset_index(drop=True)
kw_cities = cells.groupby('keyword').city.nunique()
never = sorted(set(dictionary.keyword) - set(cells.keyword))
print(f'{len(cells)} city-keyword cells | {cells.keyword.nunique()} keywords observed | never observed: {never} | '
      f'{int((kw_cities == 1).sum())} keywords retained in only one city')
check('1,855 cells and 143 observed keywords', len(cells) == 1855 and cells.keyword.nunique() == 143)

1855 city-keyword cells | 143 keywords observed | never observed: ['concert', 'fraud'] | 25 keywords retained in only one city
PASS  1,855 cells and 143 observed keywords


### 1.9 Merged 38-City Analytic Table

Combine urban indicators, segregation outcomes, and domain scores, and verify column-by-column equality against `results/analysis_design_38_metros.csv`.

In [11]:
DESIGN = pd.concat([aligned, OUT_Y.add_prefix('outcome__'), A_SCORE.add_prefix('aspect__')], axis=1)
old_design = pd.read_csv(PROJECT / 'results' / 'analysis_design_38_metros.csv', index_col='city').loc[CITIES]
numeric = old_design.select_dtypes('number').columns
check('design table equals results/analysis_design_38_metros.csv',
      list(old_design.columns) == list(DESIGN.columns)
      and np.allclose(DESIGN[numeric].astype(float), old_design[numeric], rtol=1e-10, atol=1e-9, equal_nan=True)
      and (DESIGN.drop(columns=numeric).astype(str).to_numpy() == old_design.drop(columns=numeric).astype(str).to_numpy()).all())

PASS  design table equals results/analysis_design_38_metros.csv


### 1.10 Score Indicators

Transit Score, Walking Score, and Driving Score in `census_cities.csv` are not standard 0–100 Walk Score values (their original transform is undocumented). Walking Score and Driving Score appear among the 11 archived baseline predictors; Transit Score is included only in exploratory transport checks.

In [12]:
display(aligned[['Transit Score', 'Walking Score', 'Driving Score']].describe().loc[['min', '50%', 'max']].round(2))

,Transit Score,Walking Score,Driving Score
min,0.31,0.14,30.11
50%,1.12,0.44,61.22
max,3.54,1.17,125.91


### 1.11 Save Preprocessing Tables

In [13]:
for frame, name, idx in [(crosswalk, 'city_metro_crosswalk.csv', False),
                         (coverage, 'coverage_by_city_domain.csv', False),
                         (cells, 'city_keyword_cells.csv', False),
                         (DESIGN, 'analysis_design_38_cities.csv', True),
                         (SIZE, 'corpus_size_proxy.csv', True)]:
    save(frame, 'preprocessing', name, index=idx)
print(sorted(p.name for p in (OUT / 'preprocessing').iterdir()))

['analysis_design_38_cities.csv', 'city_keyword_cells.csv', 'city_metro_crosswalk.csv', 'corpus_size_proxy.csv', 'coverage_by_city_domain.csv']


## 2. Baseline Reference Reproduction

Verify exact reproduction of all published baseline tables against the archived CSVs (`rtol <= 1e-8`).

### 2.1 Reference Models Across Six Outcomes and Leave-One-City-Out Checks

Two 38-city standardized OLS models:
- **Urban-statistics branch:** 11 civic/demographic indicators;
- **Online-opinion branch:** 12 zero-filled domain scores.

Inference uses population standard deviations (`ddof=0`), HC3 standard errors, residual-df $t$-tests, and within-model Benjamini–Hochberg adjustment across non-intercept terms. Leave-one-city-out checks fix full-sample standardization before dropping each city.

In [14]:
full_rows, loo_rows = [], []
for branch, x_raw in [('objective_archived', aligned[TABLE3_PREDICTORS]), ('online_archived', A_SCORE.fillna(0.0))]:
    for outcome in OUTCOMES:
        model, robust, x, y = fit_model(x_raw, OUT_Y[outcome])
        family = f'{branch}|{outcome}|all_{x_raw.shape[1]}_predictors'
        full_rows += coefficient_rows(model, robust, branch, outcome, family,
                                      missing_handling='archived_zero_fill' if branch == 'online_archived' else 'none')
        for i, city in enumerate(x_raw.index):
            keep = np.arange(len(x_raw)) != i
            omitted = sm.OLS(y.iloc[keep], x.iloc[keep]).fit()
            for term in model.params.index[1:]:
                loo_rows.append({'branch': branch, 'outcome': outcome, 'term': term, 'omitted_city': city,
                                 'beta_loo': float(omitted.params[term]), 'beta_full': float(model.params[term])})
FULL = add_fdr(pd.DataFrame(full_rows), ['family'])
REF = FULL[FULL.outcome.eq('archived_composite')]
expected = {'objective_archived': [0.0, .5812, .6174, .3602, .3009, .1307, .6015, -.0070, .2348, -.1384, -.2758, .3670],
            'online_archived': [0.0, -.3242, -.3741, -.3834, .0541, .1066, .1191, .1472, -.4213, -.3741, .0866, -.1861, -.2482]}
expected_se = {'objective_archived': [.087, .194, .184, .222, .241, .106, .211, .174, .128, .125, .191, .239],
               'online_archived': [.135, .151, .170, .214, .197, .167, .158, .176, .174, .183, .177, .198, .167]}
expected_p = {'objective_archived': [1., .006, .002, .117, .222, .231, .008, .968, .079, .278, .161, .137],
              'online_archived': [1., .042, .038, .085, .786, .529, .459, .410, .023, .051, .630, .356, .151]}
for branch in expected:
    b = REF[REF.branch.eq(branch)]
    check(f'{branch}: displayed coefficients, SEs, p values and R-squared reproduce',
          np.allclose(np.round(b.beta, 4), expected[branch], atol=1e-10)
          and np.allclose(np.round(b.ols_se, 3), expected_se[branch], atol=1e-10)
          and np.allclose(np.round(b.ols_p, 3), expected_p[branch], atol=1e-10)
          and round(b.r_squared.iloc[0], 3) == (.805 if branch == 'objective_archived' else .545))
online = REF[REF.branch.eq('online_archived') & REF.term.ne('const')]
check('reference Reddit model: smallest HC3 q is .209', round(online.hc3_q_bh.min(), 3) == 0.209)
ok, detail = compare_to_archive(FULL, PROJECT / 'results' / 'full_model_all_outcomes_ols_hc3_fdr.csv',
                                ['branch', 'outcome', 'term'], ['beta', 'ols_se', 'ols_p', 'hc3_se', 'hc3_p', 'hc3_q_bh', 'r_squared'])
check('both reference designs x 6 outcomes match the archive', ok, detail)
display(online[['term', 'beta', 'hc3_ci_low', 'hc3_ci_high', 'hc3_p', 'hc3_q_bh']].round(3))
save(FULL, 'original', 'reference_models_6_outcomes.csv')

PASS  objective_archived: displayed coefficients, SEs, p values and R-squared reproduce
PASS  online_archived: displayed coefficients, SEs, p values and R-squared reproduce
PASS  reference Reddit model: smallest HC3 q is .209
PASS  both reference designs x 6 outcomes match the archive   [150 rows, 0 unmatched, max |diff| 5.0e-13]


,term,beta,hc3_ci_low,hc3_ci_high,hc3_p,hc3_q_bh
73,Public Transportation,-0.324,-0.632,-0.016,0.040,0.209
74,Driving and Traffic,-0.374,-0.752,0.004,0.052,0.209
75,Economy and Income,-0.383,-0.916,0.149,0.151,0.309
76,Education,0.054,-0.414,0.523,0.814,0.814
77,Culture and Art,0.107,-0.375,0.589,0.653,0.717
78,Race/Immigration Discourse,0.119,-0.170,0.408,0.405,0.607
79,Recreation,0.147,-0.322,0.616,0.524,0.698
80,Social Problem,-0.421,-0.797,-0.046,0.029,0.209
81,Governing,-0.374,-0.787,0.039,0.074,0.223
82,Policing,0.087,-0.311,0.484,0.658,0.717


In [15]:
loo = pd.DataFrame(loo_rows)
summ = []
for (branch, outcome, term), s in loo.groupby(['branch', 'outcome', 'term'], sort=False):
    summ.append({'branch': branch, 'outcome': outcome, 'term': term, 'beta_full': s.beta_full.iloc[0],
                 'beta_loo_min': s.beta_loo.min(), 'beta_loo_max': s.beta_loo.max(),
                 'sign_consistent_omissions': int((np.sign(s.beta_loo) == np.sign(s.beta_full)).sum()), 'omissions': len(s)})
LOO_SUM = pd.DataFrame(summ)
ok, detail = compare_to_archive(LOO_SUM, PROJECT / 'results' / 'leave_one_metro_out_summary.csv', ['branch', 'outcome', 'term'],
                                ['beta_full', 'beta_loo_min', 'beta_loo_max', 'sign_consistent_omissions'])
check('leave-one-city-out summary matches the archive', ok, detail)
display(LOO_SUM[(LOO_SUM.branch == 'online_archived') & (LOO_SUM.outcome == 'archived_composite')
                & LOO_SUM.term.isin(['Public Transportation', 'Driving and Traffic', 'Social Problem'])].round(3))
save(LOO_SUM, 'original', 'leave_one_city_out_summary.csv')

PASS  leave-one-city-out summary matches the archive   [138 rows, 0 unmatched, max |diff| 1.6e-12]


,branch,outcome,term,beta_full,beta_loo_min,beta_loo_max,sign_consistent_omissions,omissions
66,online_archived,archived_composite,Public Transportation,-0.324,-0.383,-0.258,38,38
67,online_archived,archived_composite,Driving and Traffic,-0.374,-0.431,-0.308,38,38
73,online_archived,archived_composite,Social Problem,-0.421,-0.524,-0.368,38,38


### 2.2 Outcome Correlations

In [16]:
CORR = OUT_Y.corr()
display(CORR.loc[['combined_group_share', 'normalized_composite', 'equal_pairwise_mean',
                  'd_black_white', 'd_hispanic_white', 'd_asian_white'], ['archived_composite']].T.round(3))
save(CORR, 'original', 'outcome_correlations.csv', index=True)

,combined_group_share,normalized_composite,equal_pairwise_mean,d_black_white,d_hispanic_white,d_asian_white
archived_composite,0.927,0.318,0.518,0.262,0.633,0.313


### 2.3 Outcome-Side Composite Decomposition ($C = S \cdot N$)

Centering $C$ around its mean yields three exact additive components:
- **Group-share component:** $U = \bar N (S - \bar S)$
- **Normalized segregation component:** $V = \bar S (N - \bar N)$
- **Interaction component:** $W = (S - \bar S)(N - \bar N) - \overline{(S - \bar S)(N - \bar N)}$

Projecting $U, V, W$ scaled by $\mathrm{SD}(C)$ onto the fixed zero-filled 12-domain OLS design exactly decomposes every $C$ slope: $b_C = b_U + b_V + b_W$. (Note that $U$ captures demographic *population shares*, distinct from keyword *composition* $c_{ig}$ in Algorithm 1.)

In [17]:
def outcome_side_split(x_zero, C, S, N):
    """C - mean(C) = mean(N)(S - mean S) + mean(S)(N - mean N) + centred (S - mean S)(N - mean N), all on C's SD."""
    X = np.column_stack([np.ones(len(C)), zscore(x_zero).to_numpy()])
    c_sd = C.std(ddof=0)
    inter = (S - S.mean()) * (N - N.mean())
    comps = {'U_share': N.mean() * (S - S.mean()), 'V_normalized': S.mean() * (N - N.mean()), 'W_interaction': inter - inter.mean()}
    beta = {k: np.linalg.lstsq(X, (v / c_sd).to_numpy(), rcond=None)[0] for k, v in comps.items()}
    beta['C'] = np.linalg.lstsq(X, ((C - C.mean()) / c_sd).to_numpy(), rcond=None)[0]
    return pd.DataFrame(beta, index=['const'] + list(x_zero.columns))[['C', 'U_share', 'V_normalized', 'W_interaction']]


UVW = outcome_side_split(A_SCORE.fillna(0.0), OUT_Y.archived_composite, OUT_Y.combined_group_share, OUT_Y.normalized_composite)
check('U + V + W equals the original coefficient', np.allclose(UVW.iloc[:, 1:].sum(axis=1), UVW.C, atol=1e-10))
t = UVW.loc['Public Transportation'].round(3)
check('transit: -.324 = -.306 + .008 - .026', tuple(t) == (-0.324, -0.306, 0.008, -0.026), tuple(t))
old = pd.read_csv(PROJECT / 'results' / 'reference_decomposition' / 'fixed_design_decomposition_coefficients.csv').set_index('term').loc[UVW.index]
check('decomposition matches results/reference_decomposition', np.allclose(UVW.to_numpy(), old[['original_beta', 'composition_component_beta',
      'share_normalized_component_beta', 'interaction_component_beta']].to_numpy(), atol=1e-9))
display(UVW.loc[['Public Transportation', 'Driving and Traffic', 'Social Problem']].round(3))
save(UVW, 'original', 'outcome_side_decomposition.csv', index=True)

PASS  U + V + W equals the original coefficient
PASS  transit: -.324 = -.306 + .008 - .026   [(-0.324, -0.306, 0.008, -0.026)]
PASS  decomposition matches results/reference_decomposition


,C,U_share,V_normalized,W_interaction
Public Transportation,-0.324,-0.306,0.008,-0.026
Driving and Traffic,-0.374,-0.404,0.036,-0.006
Social Problem,-0.421,-0.285,-0.174,0.038


### 2.4 Original Single-Domain Model Grid (432 Models)

12 domains $\times$ 2 missing-data rules (listwise vs. zero-fill) $\times$ 3 control sets (none; density + income; density + income + race shares) $\times$ 6 outcomes, with Benjamini–Hochberg adjustment within each 12-domain family.

In [18]:
CTRL_SETS = {'none': [], 'density_income': CONTROLS, 'density_income_shares': CONTROLS + SHARES}
ORIG_VARIANT = {'none': 'unadjusted', 'density_income': 'density_income', 'density_income_shares': 'density_income_composition'}
pars_rows = []
for rule in ['available_case', 'archived_zero_fill']:
    for tag, ctrls in CTRL_SETS.items():
        variant = ORIG_VARIANT[tag]
        for outcome in OUTCOMES:
            family = f'parsimonious|{rule}|{variant}|{outcome}|12_focal_groups'
            for g in DOMAINS:
                avail = A_SCORE[g].notna() if rule == 'available_case' else pd.Series(True, index=CITY_IDX)
                x = pd.concat([A_SCORE[g].fillna(0).rename(g), aligned[ctrls]], axis=1).loc[avail]
                model, robust, _, _ = fit_model(x, OUT_Y.loc[avail, outcome])
                pars_rows += coefficient_rows(model, robust, 'parsimonious_exploratory', outcome, family, focal_group=g,
                                              variant=variant, missing_handling=rule, controls='|'.join(ctrls))
PARS = pd.DataFrame(pars_rows)
PARS = add_fdr(PARS, ['family'], PARS.term.eq(PARS.focal_group))
ok, detail = compare_to_archive(PARS, PROJECT / 'results' / 'parsimonious_all_coefficients.csv', ['family', 'focal_group', 'term'],
                                ['n', 'beta', 'hc3_se', 'hc3_p', 'hc3_q_bh'])
check('original single-domain grid matches the archive', ok, detail)
focus = PARS[(PARS.missing_handling == 'available_case') & (PARS.variant == 'density_income')
             & (PARS.outcome == 'archived_composite') & PARS.term.eq(PARS.focal_group)].set_index('term')
f3 = focus.loc[['Public Transportation', 'Driving and Traffic', 'Social Problem'], ['n', 'beta', 'hc3_q_bh']].round(3)
check('available case + density/income: transit -.170 (n 37), driving .026 (n 38), social -.307 (n 35)',
      f3.beta.tolist() == [-0.170, 0.026, -0.307] and f3.n.tolist() == [37, 38, 35])
display(f3)
save(PARS, 'original', 'single_domain_grid_original.csv')

PASS  original single-domain grid matches the archive   [1872 rows, 0 unmatched, max |diff| 4.3e-12]
PASS  available case + density/income: transit -.170 (n 37), driving .026 (n 38), social -.307 (n 35)


,n,beta,hc3_q_bh
term,,,
Public Transportation,37,-0.170,0.563
Driving and Traffic,38,0.026,0.933
Social Problem,35,-0.307,0.109


### 2.5 Original Direct Cross-Source Correlations (132 Pairs per Family)

Pairwise Pearson and Spearman correlations (available-case and zero-filled) between the 12 domain scores and 11 urban indicators, BH-adjusted across all 132 tests per family.

In [19]:
rows = []
for rule in ['available_case', 'archived_zero_fill']:
    for g in DOMAINS:
        x = A_SCORE[g] if rule == 'available_case' else A_SCORE[g].fillna(0)
        for term in TABLE3_PREDICTORS:
            pair = pd.concat([x, aligned[term]], axis=1).dropna()
            r, p = stats.pearsonr(pair.iloc[:, 0], pair.iloc[:, 1])
            rho, sp = stats.spearmanr(pair.iloc[:, 0], pair.iloc[:, 1])
            rows.append({'missing_handling': rule, 'group': g, 'indicator': term, 'n': len(pair),
                         'pearson_r': r, 'pearson_p': p, 'spearman_rho': rho, 'spearman_p': sp})
DIRECT = pd.DataFrame(rows)
for rule, idx in DIRECT.groupby('missing_handling').groups.items():
    for method in ['pearson', 'spearman']:
        DIRECT.loc[idx, f'{method}_q_bh'] = multipletests(DIRECT.loc[idx, f'{method}_p'], method='fdr_bh')[1]
ok, detail = compare_to_archive(DIRECT, PROJECT / 'results' / 'direct_cross_source_all_264.csv', ['missing_handling', 'group', 'indicator'],
                                ['n', 'pearson_r', 'pearson_p', 'spearman_rho', 'spearman_p', 'pearson_q_bh', 'spearman_q_bh'])
check('264 direct comparisons match the archive', ok, detail)
prim = DIRECT[DIRECT.missing_handling == 'available_case']
check('two Pearson and two Spearman pairs pass BH', (prim.pearson_q_bh < .05).sum() == 2 and (prim.spearman_q_bh < .05).sum() == 2)
display(prim[(prim.pearson_q_bh < .05) | (prim.spearman_q_bh < .05)]
        [['group', 'indicator', 'n', 'pearson_r', 'pearson_q_bh', 'spearman_rho', 'spearman_q_bh']].round(3))
save(DIRECT, 'original', 'direct_comparisons_264.csv')

PASS  264 direct comparisons match the archive   [264 rows, 0 unmatched, max |diff| 6.3e-12]
PASS  two Pearson and two Spearman pairs pass BH


,group,indicator,n,pearson_r,pearson_q_bh,spearman_rho,spearman_q_bh
27,Economy and Income,commute_%drive,37,0.436,0.140,0.606,0.009
40,Education,Black_White income inequality,38,0.381,0.140,0.572,0.011
75,Recreation,Driving Score,38,0.540,0.031,0.501,0.055
115,Neighborhood,commute_%drive,38,0.551,0.031,0.426,0.113


### 2.6 Behavioral Regressions

Regress driving share, carpooling share, and homeownership on the 12 zero-filled domain scores (36 non-intercept coefficients, BH-adjusted jointly).

In [20]:
beh_rows = []
for outcome in BEHAVIORS:
    model, robust, _, _ = fit_model(A_SCORE.fillna(0), aligned[outcome])
    beh_rows += coefficient_rows(model, robust, 'rq3_archived', outcome, 'RQ3_all_36_discourse_outcome_coefficients',
                                 missing_handling='archived_zero_fill')
BEH = add_fdr(pd.DataFrame(beh_rows), ['family'])
ok, detail = compare_to_archive(BEH, PROJECT / 'results' / 'rq3_all_36_coefficients_hc3_fdr.csv', ['outcome', 'term'],
                                ['beta', 'hc3_p', 'hc3_q_bh', 'r_squared'])
check('behavioral models match the archive', ok, detail)
check('none of the 36 behavioral coefficients passes BH', (BEH.hc3_q_bh.dropna() >= .05).all())
save(BEH, 'original', 'behavioral_models_36.csv')

PASS  behavioral models match the archive   [39 rows, 0 unmatched, max |diff| 4.9e-13]
PASS  none of the 36 behavioral coefficients passes BH


## 3. Algorithm 1: Decomposing Domain Scores into Tone, Composition, and Coverage

For each domain $g$, fit the unweighted two-way fixed-effects model on retained city–keyword cells $(i, k)$:

$$m_{ik} = \alpha_{ig} + \beta_k + \varepsilon_{ik}$$

- $\beta_k$ is the **keyword baseline** (how favorably or unfavorably term $k$ tends to score across cities).
- $\alpha_{ig}$ is the city's **tone** (whether city $i$ scores higher or lower than keyword baselines overall, centered to mean 0 across observed cities).
- **Composition** $c_{ig} = |\mathcal{K}_{ig}|^{-1} \sum_{k \in \mathcal{K}_{ig}} \hat\beta_k$ is the mean baseline of keywords retained in city $i$.
- **Exact identity:** $a_{ig} = \hat\alpha_{ig} + c_{ig}$ because OLS within-city residuals sum to zero.

Reliability of city tone is estimated via fixed-effects variance:

$$\rho_g = 1 - \frac{\bar v_g}{\operatorname{Var}_i(\hat\alpha_{ig})}$$

where $\bar v_g$ is the mean sampling variance of $\hat\alpha_{ig}$ under homoskedasticity; $\rho_g \le 0$ indicates that apparent city tone differences are indistinguishable from sampling noise.

In [21]:
def components(ci, ki):
    """Connected parts of the city-keyword graph (cities joined through shared keywords)."""
    uc, c2 = np.unique(ci, return_inverse=True)
    parent = np.arange(len(uc))

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    first = {}
    for c, k in zip(c2, ki):
        if k in first:
            r1, r2 = find(c), find(first[k])
            if r1 != r2:
                parent[r1] = r2
        else:
            first[k] = c
    roots = np.array([find(x) for x in range(len(uc))], dtype=int)
    return len(np.unique(roots)), uc, roots


def fe_fit(ci, ki, y):
    """Two-way fixed effects on one domain. Tones are centred over cities; keyword baselines carry the shift."""
    uc, c2 = np.unique(ci, return_inverse=True)
    uk, k2 = np.unique(ki, return_inverse=True)
    n, nc, nk = len(y), len(uc), len(uk)
    X = np.zeros((n, nc + nk - 1))
    X[np.arange(n), c2] = 1.0
    s = k2 > 0
    X[np.nonzero(s)[0], nc + k2[s] - 1] = 1.0
    th = np.linalg.lstsq(X, y, rcond=None)[0]
    alpha = th[:nc]
    shift = alpha.mean()
    beta = np.r_[0.0, th[nc:]] + shift
    cnt = np.bincount(c2, minlength=nc)
    return dict(uc=uc, uk=uk, c2=c2, k2=k2, X=X, y=y, th=th, cnt=cnt, beta=beta, tone=alpha - shift,
                comp=np.bincount(c2, weights=beta[k2], minlength=nc) / cnt,
                a=np.bincount(c2, weights=y, minlength=nc) / cnt)


def fe_connected(ci, ki, y):
    """Fit on the largest connected part (cities outside it get no tone); also report whether the graph split."""
    k, uc, roots = components(ci, ki)
    if k > 1:
        keep = np.isin(ci, uc[roots == np.bincount(roots).argmax()])
        ci, ki, y = ci[keep], ki[keep], y[keep]
    return fe_fit(ci, ki, y), k > 1


def fe_noise(f):
    """Homoskedastic sampling (co)variances of the centred tones and compositions, averaged over cities."""
    X, th, c2, k2, cnt = f['X'], f['th'], f['c2'], f['k2'], f['cnt']
    n, p = X.shape
    nc = len(f['uc'])
    e = f['y'] - X @ th
    df = n - p
    s2 = e @ e / df
    V = s2 * np.linalg.inv(X.T @ X)
    A = np.zeros((nc, p))
    A[np.arange(nc), np.arange(nc)] = 1.0
    Cm = np.zeros((nc, p))
    s = k2 > 0
    np.add.at(Cm, (c2[s], nc + k2[s] - 1), 1.0 / cnt[c2[s]])
    M = np.eye(nc) - 1.0 / nc
    avg = lambda L, R: np.trace(M @ L @ V @ R.T @ M) / (nc - 1)
    return dict(s2=s2, df=df, b_tone=avg(A, A), b_comp=avg(Cm, Cm), b_cross=avg(A, Cm))


def mde_r(n, alpha=.05, power=.8):
    """Smallest correlation detectable with the given power (two-sided, Fisher z)."""
    rs = np.linspace(.01, .99, 981)
    pw = stats.norm.sf(stats.norm.isf(alpha / 2) - np.arctanh(rs) * np.sqrt(n - 3))
    return rs[np.argmax(pw >= power)]


def measure(cells_in, domains=DOMAINS):
    """Algorithm 1 on a city-keyword cell table: signal tables (38 x 12) and per-domain fits."""
    sig = {k: pd.DataFrame(np.nan, index=CITY_IDX, columns=domains) for k in ['a', 'tone', 'comp', 'n']}
    fits = {}
    for g in domains:
        gg = cells_in[cells_in.domain == g]
        ci, ki, y = CITY_IDX.get_indexer(gg.city), gg.keyword.to_numpy(), gg.m.to_numpy(float)
        f, split = fe_connected(ci, ki, y)
        fits[g] = dict(fit=f, split=split, parts=components(ci, ki)[0])
        sig['tone'].loc[CITY_IDX[f['uc']], g] = f['tone']
        sig['comp'].loc[CITY_IDX[f['uc']], g] = f['comp']
        sig['a'][g] = gg.groupby('city').m.mean().reindex(CITY_IDX)
        sig['n'][g] = gg.groupby('city').size().reindex(CITY_IDX).fillna(0)
    return sig, fits

### 3.1 Run Algorithm 1 and Verify Additive Identity

In [22]:
SIG, FITS = measure(cells)
check('every domain graph is connected', all(v['parts'] == 1 for v in FITS.values()))
gap = float((SIG['a'] - SIG['tone'] - SIG['comp']).abs().max().max())
check('a = tone + composition exactly', gap < 1e-10, f'max error {gap:.1e}')
check('recomputed a equals the archived domain scores', np.allclose(SIG['a'], A_SCORE, rtol=0, atol=1e-12, equal_nan=True))
check('coverage n equals the retained-keyword counts', (SIG['n'].to_numpy() == N_COV.to_numpy()).all())

PASS  every domain graph is connected
PASS  a = tone + composition exactly   [max error 8.7e-15]
PASS  recomputed a equals the archived domain scores
PASS  coverage n equals the retained-keyword counts


### 3.2 Tone Reliability and Variance Decomposition (Algorithm 1, Lines 9–10)

Columns include observed city-level standard deviations (`sd_*`), noise-corrected SDs (`sd_*_corrected`), variance shares of tone, composition, and $2\operatorname{Cov}$ before and after noise correction (`share_*`), and minimum detectable correlation at 80% power (`mde_true_r_tone`).

In [23]:
rows = []
for g in DOMAINS:
    f = FITS[g]['fit']
    nz = fe_noise(f)
    va, vc, vaa = f['tone'].var(ddof=1), f['comp'].var(ddof=1), np.var(f['a'], ddof=1)
    cv = np.cov(f['tone'], f['comp'], ddof=1)[0, 1]
    ct, cc, cx = va - nz['b_tone'], vc - nz['b_comp'], cv - nz['b_cross']
    ca = ct + cc + 2 * cx
    nc = len(f['uc'])
    rel = 1 - nz['b_tone'] / va
    rows.append({'domain': g, 'cities': nc, 'cells': len(f['y']), 'keywords': len(f['uk']),
                 'keywords_per_city_median': np.median(f['cnt']), 'one_keyword_cities': int((f['cnt'] == 1).sum()),
                 'resid_sd': np.sqrt(nz['s2']), 'sd_score': np.sqrt(vaa),
                 'sd_tone': np.sqrt(va), 'sd_tone_corrected': np.sqrt(max(ct, 0)),
                 'sd_comp': np.sqrt(vc), 'sd_comp_corrected': np.sqrt(max(cc, 0)),
                 'reliability_tone': rel, 'reliability_comp': 1 - nz['b_comp'] / vc,
                 'share_tone_raw': va / vaa, 'share_comp_raw': vc / vaa, 'share_2cov_raw': 2 * cv / vaa,
                 'share_tone_corrected': ct / ca, 'share_comp_corrected': cc / ca, 'share_2cov_corrected': 2 * cx / ca,
                 'mde_true_r_tone': mde_r(nc) / np.sqrt(rel) if rel > 0 else np.inf})
REL = pd.DataFrame(rows).set_index('domain')
tr = REL.loc['Public Transportation']
check('transit tone reliability .277 and corrected SD .071 (pre-test values)',
      round(tr.reliability_tone, 3) == 0.277 and round(tr.sd_tone_corrected, 3) == 0.071)
print('domains with tone reliability <= 0:', REL.index[REL.reliability_tone <= 0].tolist())
print('domains where corrected composition variance exceeds corrected tone variance:',
      int((REL.sd_comp_corrected > REL.sd_tone_corrected).sum()), 'of 12')
display(REL[['cities', 'keywords_per_city_median', 'sd_score', 'sd_tone', 'sd_tone_corrected', 'sd_comp_corrected',
             'reliability_tone', 'share_tone_corrected', 'share_comp_corrected', 'share_2cov_corrected', 'mde_true_r_tone']].round(3))

PASS  transit tone reliability .277 and corrected SD .071 (pre-test values)
domains with tone reliability <= 0: ['Recreation', 'Social Problem', 'Governing', 'Policing']
domains where corrected composition variance exceeds corrected tone variance: 10 of 12


,cities,keywords_per_city_median,sd_score,sd_tone,sd_tone_corrected,sd_comp_corrected,reliability_tone,share_tone_corrected,share_comp_corrected,share_2cov_corrected,mde_true_r_tone
domain,,,,,,,,,,,
Public Transportation,37,5.0,0.264,0.135,0.071,0.263,0.277,0.088,1.217,-0.305,0.850
Driving and Traffic,38,6.0,0.187,0.103,0.056,0.141,0.296,0.115,0.715,0.170,0.812
Economy and Income,37,4.0,0.213,0.193,0.131,0.072,0.464,0.671,0.201,0.128,0.657
Education,38,2.0,0.213,0.164,0.080,0.107,0.240,0.257,0.455,0.289,0.903
Culture and Art,37,3.0,0.240,0.150,0.086,0.154,0.330,0.173,0.553,0.274,0.778
Race/Immigration Discourse,23,2.0,0.363,0.242,0.116,0.254,0.229,0.146,0.705,0.149,1.163
Recreation,38,3.5,0.220,0.101,0.000,0.196,-0.354,-0.102,1.101,0.002,inf
Social Problem,35,2.0,0.153,0.134,0.000,0.111,-0.322,-6.902,14.850,-6.948,inf
Governing,34,5.0,0.211,0.151,0.000,0.177,-0.419,-0.746,2.445,-0.699,inf


### 3.3 Concrete Example: Portland vs. Austin in Public Transportation

Portland's tone is simply the mean deviation of its retained public-transit keywords relative to each keyword's cross-city baseline.

In [24]:
baselines = pd.concat([pd.DataFrame({'domain': g, 'keyword': FITS[g]['fit']['uk'], 'baseline_beta': FITS[g]['fit']['beta']})
                       for g in DOMAINS], ignore_index=True)
baselines = baselines.merge(cells.groupby('keyword').agg(cities=('city', 'size'), raw_mean=('m', 'mean')).reset_index(), on='keyword')
ex = cells[(cells.domain == 'Public Transportation') & (cells.city == 'Portland')].merge(baselines[['keyword', 'baseline_beta']], on='keyword')
ex['difference'] = ex.m - ex.baseline_beta
display(ex.sort_values('baseline_beta').round(3))
display(pd.DataFrame({k: SIG[k].loc[['Portland', 'Austin'], 'Public Transportation'] for k in ['n', 'a', 'comp', 'tone']}).round(3))

,city,domain,keyword,m,baseline_beta,difference
3,Portland,Public Transportation,bus,-0.455,0.166,-0.621
0,Portland,Public Transportation,commute,0.080,0.265,-0.185
2,Portland,Public Transportation,transportation,0.912,0.688,0.224
1,Portland,Public Transportation,transit,0.914,0.756,0.157


,n,a,comp,tone
city,,,,
Portland,4.0,0.363,0.469,-0.106
Austin,10.0,0.109,-0.043,0.153


### 3.4 Random-Effects Variance Components (Algorithm 3, Line 7)

Crossed random-effects specification $m_{ik} = \mu_g + u_{ig} + v_k + \varepsilon_{ik}$ fit via REML. Best linear unbiased predictions (BLUPs) shrink low-coverage city estimates toward zero and partially allocate city-specific proper-noun effects back toward tone.

**Selection Rule**
BLUP tone is substituted only where the boundary-corrected REML likelihood-ratio test of $H_0: \sigma^2_u = 0$ yields $p_{\text{boundary}} < 0.20$:
- Under a parameter-space boundary null, the LRT distribution is approximated by a $0.5\chi^2_0 + 0.5\chi^2_1$ mixture (Self & Liang 1987; Stram & Lee 1994), so $p_{\text{boundary}} = \frac{1}{2} P(\chi^2_1 \ge \text{LRT})$ (approximate under crossed cities and keywords).
- The $\alpha = 0.20$ threshold aligns with Matuschek et al. (2017), who demonstrate that standard $\alpha = 0.05$ over-penalizes random-effects selection.

Eight domains pass $p_{\text{boundary}} < 0.20$; Governing, Social Problem, Recreation, and City Development fail and retain fixed-effects tone in the REML robustness check (`Race/Immigration Discourse` is near the threshold at $p = 0.192$).

In [25]:
def reml(ci, ki, y, fix_city_zero=False):
    """Crossed random effects m_ik = mu + u_i + v_k + e_ik by REML; returns variances, BLUP of u and the criterion."""
    n = len(y)
    uc, c2 = np.unique(ci, return_inverse=True)
    uk, k2 = np.unique(ki, return_inverse=True)
    Zu = np.zeros((n, len(uc)))
    Zu[np.arange(n), c2] = 1
    Zv = np.zeros((n, len(uk)))
    Zv[np.arange(n), k2] = 1
    Ku, Kv, I, one = Zu @ Zu.T, Zv @ Zv.T, np.eye(n), np.ones(n)

    def unpack(t):
        return (0.0 if fix_city_zero else np.exp(t[0])), np.exp(t[-2]), np.exp(t[-1])

    def parts(t):
        su, sv, se = unpack(t)
        cf = cho_factor(su * Ku + sv * Kv + se * I, lower=True)
        vi1 = cho_solve(cf, one)
        mu = (one @ cho_solve(cf, y)) / (one @ vi1)
        return cf, y - mu, one @ vi1

    def nll(t):
        cf, r, xvx = parts(t)
        return 0.5 * (2 * np.log(np.diag(cf[0])).sum() + np.log(xvx) + r @ cho_solve(cf, r))

    v0 = np.log(np.var(y) / 3)
    starts = [[v0, v0], [v0 - 2, v0]] if fix_city_zero else [[v0, v0, v0], [v0 - 2, v0, v0], [v0, v0 - 2, v0]]
    best = min((optimize.minimize(nll, s, method='L-BFGS-B', bounds=[(-14, 3)] * len(s)) for s in starts), key=lambda o: o.fun)
    su, sv, se = unpack(best.x)
    cf, r, _ = parts(best.x)
    return dict(uc=uc, su=su, sv=sv, se=se, u=su * Zu.T @ cho_solve(cf, r), nll=best.fun, converged=bool(best.success),
                city_at_boundary=(not fix_city_zero) and best.x[0] < -13.5)


RE, rows = {}, []
for g in DOMAINS:
    gg = cells[cells.domain == g]
    ci, ki, y = CITY_IDX.get_indexer(gg.city), gg.keyword.to_numpy(), gg.m.to_numpy(float)
    fit, null = reml(ci, ki, y), reml(ci, ki, y, fix_city_zero=True)
    LR = max(2 * (null['nll'] - fit['nll']), 0.0)
    u = pd.Series(fit['u'] - fit['u'].mean(), index=CITY_IDX[fit['uc']]).reindex(CITY_IDX)
    eligible = (not fit['city_at_boundary']) and 0.5 * stats.chi2.sf(LR, 1) < RE_LRT_ALPHA
    RE[g] = dict(u=u, eligible=bool(eligible))
    fe_tone = SIG['tone'][g]
    rows.append({'domain': g, 'converged': fit['converged'], 'var_city': fit['su'], 'sd_city': np.sqrt(fit['su']),
                 'var_keyword': fit['sv'], 'var_resid': fit['se'],
                 'keyword_over_city_variance': fit['sv'] / fit['su'] if fit['su'] > 0 else np.inf,
                 'LR_city_variance_zero': LR, 'p_boundary': 0.5 * stats.chi2.sf(LR, 1),
                 'r_blup_with_fe_tone': u.corr(fe_tone) if u.std() > 1e-8 else np.nan,
                 'sd_blup_over_sd_fe': u.std() / fe_tone.std(), 'use_blup_tone': bool(eligible)})
RE_TAB = pd.DataFrame(rows).set_index('domain')
display(RE_TAB.round(4))
print('BLUP tone used in:', [g for g in DOMAINS if RE[g]['eligible']])
print('not used:', [g for g in DOMAINS if not RE[g]['eligible']])

,converged,var_city,sd_city,var_keyword,var_resid,keyword_over_city_variance,LR_city_variance_zero,p_boundary,r_blup_with_fe_tone,sd_blup_over_sd_fe,use_blup_tone
domain,,,,,,,,,,,
Public Transportation,True,0.0045,0.0674,0.3640,0.0431,80.0559,4.6372,0.0156,0.8405,0.2864,True
Driving and Traffic,True,0.0045,0.0674,0.3331,0.0377,73.3656,5.0658,0.0122,0.9893,0.4104,True
Economy and Income,True,0.0065,0.0804,0.1202,0.0637,18.5910,2.3534,0.0625,0.8574,0.2308,True
Education,True,0.0073,0.0855,0.4020,0.0397,54.9515,1.0988,0.1473,0.9843,0.2718,True
Culture and Art,True,0.0068,0.0827,0.0606,0.0300,8.8600,2.9306,0.0435,0.9359,0.3294,True
Race/Immigration Discourse,True,0.0114,0.1066,0.2466,0.0763,21.7153,0.7561,0.1923,0.9144,0.2234,True
Recreation,True,0.0014,0.0368,0.2018,0.0358,148.9120,0.4210,0.2582,0.8977,0.1276,False
Social Problem,True,0.0004,0.0209,0.0684,0.0374,156.0746,0.0416,0.4192,0.7089,0.0287,False
Governing,True,0.0000,0.0009,0.1157,0.0831,139159.0602,0.0000,0.5000,0.7597,0.0000,False


BLUP tone used in: ['Public Transportation', 'Driving and Traffic', 'Economy and Income', 'Education', 'Culture and Art', 'Race/Immigration Discourse', 'Policing', 'Neighborhood']
not used: ['Recreation', 'Social Problem', 'Governing', 'City Development']


### 3.5 Save Measurement Outputs

In [26]:
for k, frame in SIG.items():
    save(frame, 'measurement', f'signal_{k}.csv', index=True)
save(REL.reset_index(), 'measurement', 'reliability_and_variance_shares.csv')
save(baselines, 'measurement', 'keyword_baselines.csv')
save(RE_TAB.reset_index(), 'measurement', 'random_effects_variance_components.csv')
print(sorted(p.name for p in (OUT / 'measurement').iterdir()))

['keyword_baselines.csv', 'random_effects_variance_components.csv', 'reliability_and_variance_shares.csv', 'signal_a.csv', 'signal_comp.csv', 'signal_n.csv', 'signal_tone.csv']


## 4. Design Diagnostics (Outcome-Blind Pre-Tests)

All checks in this section use either outcome-free network/coverage quantities or permuted/simulated outcomes to validate inference calibration. Toggle `RUN_DESIGN_DIAGNOSTICS = False` to skip without affecting downstream sections. Control specifications add $\log s_i$ to all three baseline control sets.

### 4.1 Coverage, Corpus Size, and Multicollinearity (VIF)

In [27]:
if RUN_DESIGN_DIAGNOSTICS:
    rows = []
    for g in DOMAINS:
        n_g = SIG['n'][g]
        row = {'domain': g, 'n_range': f'{int(n_g.min())}–{int(n_g.max())}', 'r_n_log_s': n_g.corr(SIZE.log_size),
               'R2_n_on_log_s_log_pop': sm.OLS(n_g, sm.add_constant(SIZE[['log_size', 'log_pop']])).fit().rsquared}
        for tag, ctrls in CTRL_SETS.items():
            for sname in ['a', 'tone', 'n']:
                x = SIG[sname][g]
                ok = x.notna()
                Z = pd.concat([aligned[ctrls], SIZE.log_size], axis=1).loc[ok]
                row[f'VIF_{sname}_{tag}'] = 1 / (1 - sm.OLS(x[ok], sm.add_constant(Z)).fit().rsquared)
        rows.append(row)
    DIAG_COV = pd.DataFrame(rows).set_index('domain')
    display(DIAG_COV.round(2))
    save(DIAG_COV.reset_index(), 'diagnostics', 'coverage_size_and_vif.csv')
else:
    print('skipped')

,n_range,r_n_log_s,R2_n_on_log_s_log_pop,VIF_a_none,VIF_tone_none,VIF_n_none,VIF_a_density_income,VIF_tone_density_income,VIF_n_density_income,VIF_a_density_income_shares,VIF_tone_density_income_shares,VIF_n_density_income_shares
domain,,,,,,,,,,,,
Public Transportation,0–15,0.80,0.64,1.45,1.00,2.74,1.47,1.10,3.08,1.54,1.33,3.37
Driving and Traffic,2–8,0.77,0.60,1.16,1.02,2.44,1.23,1.04,2.75,1.30,1.08,4.71
Economy and Income,0–11,0.89,0.79,1.34,1.17,4.84,1.85,1.52,5.47,2.07,1.72,5.80
Education,1–4,0.38,0.14,1.05,1.00,1.16,1.28,1.29,1.32,1.96,1.61,1.60
Culture and Art,0–5,0.53,0.29,1.02,1.06,1.38,1.23,1.21,1.51,1.37,1.43,1.68
Race/Immigration Discourse,0–11,0.70,0.57,1.00,1.03,1.94,1.01,1.07,2.15,1.12,1.09,2.81
Recreation,1–9,0.55,0.31,1.49,1.11,1.44,1.56,1.22,1.66,1.72,1.37,1.78
Social Problem,0–15,0.69,0.49,1.01,1.03,1.93,1.05,1.06,2.42,1.25,1.16,2.94
Governing,0–11,0.84,0.74,1.39,1.00,3.36,1.46,1.07,4.36,2.11,1.49,6.17


### 4.2 Bipartite Graph Connectivity Under Keyword Removal

Verify whether the city–keyword bipartite graph stays connected and retains all cities under:
- single-keyword removal;
- dropping all single-city keywords;
- random 10%, 20%, and 30% removal draws.

The 30% per-domain screening cap derives from preserving connected identification.

In [28]:
if RUN_DESIGN_DIAGNOSTICS:
    single = set(kw_cities[kw_cities == 1].index)
    rows = []
    for g in DOMAINS:
        gg = cells[cells.domain == g]
        ci, ki = CITY_IDX.get_indexer(gg.city), gg.keyword.to_numpy()
        kws, ncity = np.unique(ki), len(np.unique(ci))
        sel = ~np.isin(ki, list(single))
        row = {'domain': g, 'observed_keywords': len(kws),
               'drop_one_splits_graph': sum(components(ci[ki != k], ki[ki != k])[0] > 1 for k in kws),
               'drop_one_loses_a_city': sum(len(np.unique(ci[ki != k])) < ncity for k in kws),
               'no_single_city_keywords_parts': components(ci[sel], ki[sel])[0],
               'no_single_city_keywords_cities_lost': ncity - len(np.unique(ci[sel]))}
        for share in (0.1, 0.2, 0.3):
            con, lost = [], []
            for _ in range(100 if QUICK else 500):
                sel = np.isin(ki, kws[rng.random(len(kws)) >= share])
                con.append(bool(sel.any()) and components(ci[sel], ki[sel])[0] == 1)
                lost.append(ncity - len(np.unique(ci[sel])))
            row[f'random_{int(share * 100)}pct_connected'] = np.mean(con)
            row[f'random_{int(share * 100)}pct_cities_lost'] = np.mean(lost)
        rows.append(row)
    DIAG_DROP = pd.DataFrame(rows).set_index('domain')
    display(DIAG_DROP.round(2))
    save(DIAG_DROP.reset_index(), 'diagnostics', 'keyword_removal_identification.csv')
else:
    print('skipped')

,observed_keywords,drop_one_splits_graph,drop_one_loses_a_city,no_single_city_keywords_parts,no_single_city_keywords_cities_lost,random_10pct_connected,random_10pct_cities_lost,random_20pct_connected,random_20pct_cities_lost,random_30pct_connected,random_30pct_cities_lost
domain,,,,,,,,,,,
Public Transportation,22,0,2,1,0,1.00,0.42,1.00,0.88,1.00,1.43
Driving and Traffic,11,0,0,1,0,1.00,0.01,1.00,0.11,1.00,0.23
Economy and Income,13,0,1,1,0,1.00,0.51,1.00,1.01,1.00,2.13
Education,5,0,1,1,0,0.99,0.54,0.97,1.70,0.95,3.09
Culture and Art,6,0,1,1,0,1.00,0.82,1.00,1.92,0.99,4.22
Race/Immigration Discourse,13,0,3,1,0,1.00,0.83,1.00,1.79,1.00,2.66
Recreation,14,0,1,1,0,1.00,0.31,1.00,0.92,1.00,2.12
Social Problem,19,0,1,1,0,1.00,1.32,1.00,3.08,1.00,4.80
Governing,15,0,3,1,0,1.00,0.64,1.00,1.26,1.00,2.30


### 4.3 Permutation Check: Familywise Size Under the Null

Holding city signals and controls fixed while permuting urban outcomes across cities, verify single-test rejection near 0.05 and the BH familywise false-discovery rate across 12-domain families.

In [29]:
def hc3_p_many(X, Y, j=1):
    """HC3 + t p-values for coefficient j; one column of Y per permutation."""
    n, k = X.shape
    XtXi = np.linalg.inv(X.T @ X)
    L = XtXi @ X.T
    B = L @ Y
    E = Y - X @ B
    h = np.einsum('ij,jk,ik->i', X, XtXi, X)
    se = np.sqrt((((L[j] / (1 - h))[:, None] * E) ** 2).sum(0))
    return 2 * stats.t.sf(np.abs(B[j] / se), n - k)


def bh_min(P):
    """Smallest Benjamini-Hochberg q in each row."""
    k = P.shape[1]
    return (np.sort(P, 1) * k / np.arange(1, k + 1)).min(1)


def signal(sig, rule, g):
    return {'a_available': lambda: sig['a'][g], 'a_zero_fill': lambda: sig['a'][g].fillna(0.0),
            'tone': lambda: sig['tone'][g], 'coverage_n': lambda: sig['n'][g]}[rule]()


SIGNAL_RULES = ['a_available', 'a_zero_fill', 'tone', 'coverage_n']
if RUN_DESIGN_DIAGNOSTICS:
    NPERM = 200 if QUICK else 2000
    perms = np.array([rng.permutation(len(CITIES)) for _ in range(NPERM)])
    rows = []
    for tag, ctrls in CTRL_SETS.items():
        Zall = pd.concat([aligned[ctrls], SIZE.log_size], axis=1)
        for rule in SIGNAL_RULES:
            rej, fam = [], []
            for oc in OUTCOMES:
                Yp = OUT_Y[oc].to_numpy()[perms]
                P = np.empty((NPERM, len(DOMAINS)))
                for j, g in enumerate(DOMAINS):
                    x = signal(SIG, rule, g)
                    ok = x.notna().to_numpy()
                    X = np.column_stack([np.ones(ok.sum()), zscore(x[ok]), zscore(Zall[ok])])
                    P[:, j] = hc3_p_many(X, Yp[:, ok].T)
                rej.append((P < .05).mean(0))
                fam.append((bh_min(P) < .05).mean())
            rej = np.concatenate(rej)
            rows.append({'controls': tag + ' + log s', 'signal': rule, 'reject_min': rej.min(), 'reject_median': np.median(rej),
                         'reject_max': rej.max(), 'bh12_any_min': min(fam), 'bh12_any_max': max(fam)})
    DIAG_PERM = pd.DataFrame(rows)
    print(f'{NPERM} permutations; each row pools 6 outcomes x 12 domains')
    display(DIAG_PERM.round(3))
    save(DIAG_PERM, 'diagnostics', 'permutation_dry_run.csv')
else:
    print('skipped')

2000 permutations; each row pools 6 outcomes x 12 domains


,controls,signal,reject_min,reject_median,reject_max,bh12_any_min,bh12_any_max
0,none + log s,a_available,0.039,0.052,0.070,0.075,0.094
1,none + log s,a_zero_fill,0.041,0.052,0.082,0.075,0.106
2,none + log s,tone,0.040,0.051,0.073,0.071,0.096
3,none + log s,coverage_n,0.042,0.054,0.068,0.070,0.096
4,density_income + log s,a_available,0.033,0.046,0.066,0.055,0.068
5,density_income + log s,a_zero_fill,0.032,0.048,0.076,0.066,0.088
6,density_income + log s,tone,0.030,0.045,0.062,0.056,0.078
7,density_income + log s,coverage_n,0.035,0.049,0.062,0.061,0.078
8,density_income_shares + log s,a_available,0.025,0.037,0.051,0.040,0.052
9,density_income_shares + log s,a_zero_fill,0.026,0.040,0.072,0.050,0.066


### 4.4 Bootstrap Confidence Interval Calibration

Because tone is estimated first in Algorithm 1, city bootstrap intervals re-estimate Algorithm 1 inside each resample. Simulated outcomes verify coverage under null differences and power under composition-driven differences ($\gamma = -0.35$).

In [30]:
DA = {}
for g in DOMAINS:
    gg = cells[cells.domain == g]
    kws = np.array(sorted(gg.keyword.unique()))
    ci = CITY_IDX.get_indexer(gg.city)
    DA[g] = dict(ci=ci, ki=pd.Index(kws).get_indexer(gg.keyword), y=gg.m.to_numpy(float), kws=kws,
                 rows=[np.nonzero(ci == c)[0] for c in range(len(CITIES))])


def resample_domain(g, draw):
    """Rows of the resampled cities; a city drawn twice becomes two cities."""
    parts = [(j, DA[g]['rows'][c]) for j, c in enumerate(draw) if len(DA[g]['rows'][c])]
    ci = np.concatenate([np.full(len(r), j) for j, r in parts])
    idx = np.concatenate([r for _, r in parts])
    return ci, DA[g]['ki'][idx], DA[g]['y'][idx]


def std_coefs(y, X):
    """Standardized OLS slopes (population SD), intercept dropped; NaN if a column is constant."""
    X = np.asarray(X, float).reshape(len(y), -1)
    y = np.asarray(y, float)
    sd = X.std(0)
    if (sd == 0).any() or y.std() == 0:
        return np.full(X.shape[1], np.nan)
    Xs = (X - X.mean(0)) / sd
    return np.linalg.lstsq(np.column_stack([np.ones(len(y)), Xs]), (y - y.mean()) / y.std(), rcond=None)[0][1:]


if RUN_DESIGN_DIAGNOSTICS:
    g = 'Public Transportation'
    f = FITS[g]['fit']
    av = f['uc']
    NC = len(CITIES)
    Z3 = np.column_stack([aligned[CTRL_SETS['density_income_shares']].to_numpy(float), SIZE.log_size.to_numpy(float)])
    Z0 = SIZE.log_size.to_numpy(float)[:, None]
    comp_z = (f['comp'] - f['comp'].mean()) / f['comp'].std()
    NSIM, BCAL, GAMMA = (20, 100, -0.35) if QUICK else (200, 500, -0.35)
    scen = [('no association, controls set 3 + log s', 0, Z3), ('composition-driven, controls set 3 + log s', 1, Z3),
            ('composition-driven, log s only', 1, Z0)]
    res = {s[0]: [] for s in scen}
    t0 = time.time()
    for _ in range(NSIM):
        Ys = [rng.standard_normal(NC), rng.standard_normal(NC)]
        Ys[1][av] = GAMMA * comp_z + np.sqrt(1 - GAMMA ** 2) * rng.standard_normal(len(av))
        point = {k: std_coefs(Ys[w][av], np.column_stack([f['a'], Z[av]]))[0] - std_coefs(Ys[w][av], np.column_stack([f['tone'], Z[av]]))[0]
                 for k, w, Z in scen}
        boots = {k: np.empty(BCAL) for k, _, _ in scen}
        for b in range(BCAL):
            draw = rng.integers(0, NC, NC)
            ff, _ = fe_connected(*resample_domain(g, draw))
            src = draw[ff['uc']]
            for k, w, Z in scen:
                boots[k][b] = (std_coefs(Ys[w][src], np.column_stack([ff['a'], Z[src]]))[0]
                               - std_coefs(Ys[w][src], np.column_stack([ff['tone'], Z[src]]))[0])
        for k, _, _ in scen:
            lo, hi = np.nanpercentile(boots[k], [2.5, 97.5])
            res[k].append((point[k], lo, hi))
    rows = []
    for k, _, _ in scen:
        r_ = np.array(res[k])
        covers = ((r_[:, 1] <= 0) & (r_[:, 2] >= 0)).mean()
        rows.append({'scenario': k, 'mean_difference': r_[:, 0].mean(), 'mean_interval_width': (r_[:, 2] - r_[:, 1]).mean(),
                     'covers_0': covers, 'excludes_0': 1 - covers})
    DIAG_BOOT = pd.DataFrame(rows)
    print(f'{NSIM} simulated outcomes x {BCAL} resamples, {time.time() - t0:.0f} s')
    display(DIAG_BOOT.round(3))
    save(DIAG_BOOT, 'diagnostics', 'bootstrap_calibration.csv')
else:
    print('skipped')

200 simulated outcomes x 500 resamples, 75 s


,scenario,mean_difference,mean_interval_width,covers_0,excludes_0
0,"no association, controls set 3 + log s",0.029,0.962,0.960,0.040
1,"composition-driven, controls set 3 + log s",-0.350,0.925,0.625,0.375
2,"composition-driven, log s only",-0.386,0.777,0.500,0.500


## 5. Zero-Effect Retention Simulation (Algorithm 3, Lines 9–11)

Could retention selection driven solely by corpus size ($s_i$) create apparent coverage–size correlations even when cities share zero tone differences?

1. **Fit the Poisson retention model:** assume effective scored-mention counts follow $Y_{ik} \sim \text{Poisson}(\exp(\phi_k + \theta \log s_i))$, observed iff $Y_{ik} > 10$, estimating $\theta$ via profile likelihood on a grid.
2. **Simulate 500 null worlds:** draw retained cells from the fitted retention probabilities, generate scores as $\hat\beta_k + \text{noise}$ (zero true city tone variation), recompute $a, c, \hat\alpha, n$, and compare observed $r(n_{ig}, \log s_i)$ against the simulated 5th percentile.

In [31]:
KW = np.array(sorted(cells.keyword.unique()))
kw_dom = cells.drop_duplicates('keyword').set_index('keyword').domain.reindex(KW).to_numpy()
PASS = np.zeros((len(CITIES), len(KW)), bool)
PASS[CITY_IDX.get_indexer(cells.city), pd.Index(KW).get_indexer(cells.keyword)] = True
xs = SIZE.log_size.to_numpy()


def kw_loglik(phi, theta, kept):
    lam = np.exp(phi + theta * xs)
    return np.where(kept, stats.poisson.logsf(RETENTION_THRESHOLD, lam), stats.poisson.logcdf(RETENTION_THRESHOLD, lam)).sum()


def profile(theta):
    total, phis = 0.0, np.empty(len(KW))
    for j in range(len(KW)):
        o = optimize.minimize_scalar(lambda ph: -kw_loglik(ph, theta, PASS[:, j]), bounds=(-60, 40), method='bounded')
        phis[j] = o.x
        total -= o.fun
    return total, phis


t0 = time.time()
theta_grid = np.arange(0, 4.0001, THETA_GRID_STEP)
with warnings.catch_warnings(), np.errstate(all='ignore'):
    warnings.simplefilter('ignore')
    prof = [profile(t) for t in theta_grid]
ll = np.array([p[0] for p in prof])
ib = int(ll.argmax())
THETA, PHI = theta_grid[ib], prof[ib][1]
LR_THETA = 2 * (ll[ib] - ll[0])
P_HAT = stats.poisson.sf(RETENTION_THRESHOLD, np.exp(PHI[None, :] + THETA * xs[:, None]))
print(f'theta = {THETA:.2f} (grid edge 4 would mean unbounded) | LR against theta = 0: {LR_THETA:.1f} | {time.time() - t0:.0f} s')
print('fit, r(expected coverage, observed coverage):',
      ', '.join(f'{g[:12]} {np.corrcoef(P_HAT[:, kw_dom == g].sum(1), SIG["n"][g])[0, 1]:.2f}' for g in DOMAINS))

theta = 0.60 (grid edge 4 would mean unbounded) | LR against theta = 0: 1164.3 | 11 s
fit, r(expected coverage, observed coverage): Public Trans 0.83, Driving and  0.78, Economy and  0.90, Education 0.42, Culture and  0.51, Race/Immigra 0.77, Recreation 0.51, Social Probl 0.79, Governing 0.86, Policing 0.84, Neighborhood 0.78, City Develop 0.90


In [32]:
BETA = np.array([dict(zip(FITS[g]['fit']['uk'], FITS[g]['fit']['beta']))[k] for k, g in zip(KW, kw_dom)])
S2 = {g: fe_noise(FITS[g]['fit'])['s2'] for g in DOMAINS}
observed = {}
for g in DOMAINS:
    f = FITS[g]['fit']
    xi = xs[f['uc']]
    observed[g] = [np.corrcoef(v, xi)[0, 1] for v in (f['a'], f['comp'], f['tone'])] + [np.corrcoef(SIG['n'][g], xs)[0, 1]]
sims = {g: [] for g in DOMAINS}
nsplit = {g: 0 for g in DOMAINS}
null_rng = np.random.default_rng(SEED + 4)
t0 = time.time()
for _ in range(R_NULL):
    kept = null_rng.random(P_HAT.shape) < P_HAT
    for g in DOMAINS:
        cols = np.nonzero(kw_dom == g)[0]
        ci, kj = np.nonzero(kept[:, cols])
        ki = cols[kj]
        if len(np.unique(ci)) < 3:
            sims[g].append([np.nan] * 4)
            continue
        y = BETA[ki] + null_rng.normal(0, np.sqrt(S2[g]), len(ki))
        ff, split = fe_connected(ci, ki, y)
        nsplit[g] += split
        xi = xs[ff['uc']]
        with np.errstate(invalid='ignore', divide='ignore'):
            sims[g].append([np.corrcoef(v, xi)[0, 1] for v in (ff['a'], ff['comp'], ff['tone'])]
                           + [np.corrcoef(np.bincount(ci, minlength=len(CITIES)), xs)[0, 1]])
rows = []
for g in DOMAINS:
    S_ = np.array(sims[g])
    row = {'domain': g}
    for j, lab in enumerate(['score', 'composition', 'tone', 'coverage']):
        lo, mid, hi = np.nanpercentile(S_[:, j], [5, 50, 95])
        row[f'r({lab}, log s) observed'] = observed[g][j]
        row[f'r({lab}, log s) null median'] = mid
        row[f'r({lab}, log s) null 5%'] = lo
        row[f'r({lab}, log s) null 95%'] = hi
    row['coverage: share of null runs <= observed'] = float(np.nanmean(S_[:, 3] <= observed[g][3]))
    row['split_runs'] = nsplit[g]
    rows.append(row)
NULL_SIM = pd.DataFrame(rows).set_index('domain')
beyond = NULL_SIM['r(coverage, log s) observed'] < NULL_SIM['r(coverage, log s) null 5%']
print(f'{R_NULL} null runs, {time.time() - t0:.0f} s')
print('domains whose coverage varies beyond what corpus size predicts:', int(beyond.sum()), beyond[beyond].index.tolist())
display(NULL_SIM[[c for c in NULL_SIM.columns if 'score' in c or 'coverage' in c]].round(3))
save(NULL_SIM.reset_index(), 'diagnostics', 'null_simulation.csv')

500 null runs, 2 s
domains whose coverage varies beyond what corpus size predicts: 8 ['Public Transportation', 'Education', 'Culture and Art', 'Race/Immigration Discourse', 'Recreation', 'Social Problem', 'Governing', 'Policing']


,"r(score, log s) observed","r(score, log s) null median","r(score, log s) null 5%","r(score, log s) null 95%","r(coverage, log s) observed","r(coverage, log s) null median","r(coverage, log s) null 5%","r(coverage, log s) null 95%",coverage: share of null runs <= observed
domain,,,,,,,,,
Public Transportation,-0.556,-0.669,-0.785,-0.455,0.797,0.896,0.856,0.928,0.000
Driving and Traffic,0.374,0.297,-0.121,0.546,0.768,0.830,0.758,0.891,0.080
Economy and Income,-0.504,-0.165,-0.452,0.147,0.891,0.901,0.862,0.933,0.282
Education,-0.223,-0.234,-0.463,0.055,0.376,0.582,0.426,0.725,0.012
Culture and Art,0.143,0.498,0.284,0.664,0.526,0.828,0.757,0.888,0.000
Race/Immigration Discourse,0.032,-0.127,-0.451,0.225,0.697,0.814,0.751,0.858,0.002
Recreation,-0.574,-0.639,-0.769,-0.449,0.551,0.851,0.799,0.895,0.000
Social Problem,0.119,0.251,-0.003,0.500,0.694,0.842,0.795,0.887,0.000
Governing,0.531,0.442,0.134,0.657,0.838,0.909,0.872,0.938,0.002


## 6. Keyword Screen (Algorithm 3, Line 8)

Load independent double codings from `data/keyword_screen_codes.csv` (`keep` / `drop`) and written protocol `data/keyword_screen_rules.md`, export the blank template (`results/www_short/keyword_screen/keyword_screen_TEMPLATE.csv`), verify all 145 keywords, compute Cohen's κ, enforce the 30% per-domain removal cap, verify bipartite graph connectivity, and record SHA-256 digests when `SCREEN_FROZEN = True`.

In [33]:
tmpl = dictionary.copy()
tmpl['cities_retaining'] = tmpl.keyword.map(kw_cities).fillna(0).astype(int)
tmpl['example_cities'] = tmpl.keyword.map(cells.groupby('keyword').city.agg(lambda s: '|'.join(sorted(s)[:6]))).fillna('')
for col in ['coder_A', 'coder_B', 'final_decision', 'rule_applied', 'note']:
    tmpl[col] = ''
save(tmpl, 'keyword_screen', 'keyword_screen_TEMPLATE.csv')


def cohen_kappa(a, b):
    cats = sorted(set(a) | set(b))
    po = np.mean(np.asarray(a) == np.asarray(b))
    pe = sum(np.mean(np.asarray(a) == c) * np.mean(np.asarray(b) == c) for c in cats)
    return (po - pe) / (1 - pe) if pe < 1 else np.nan


def screen_report(keep):
    """Per-domain share removed, graph connectivity and cities lost under a kept-keyword set."""
    rows = []
    for g in DOMAINS:
        listed = dictionary[dictionary.domain == g].keyword
        gg = cells[(cells.domain == g) & cells.keyword.isin(keep)]
        ci, ki = CITY_IDX.get_indexer(gg.city), gg.keyword.to_numpy()
        rows.append({'domain': g, 'listed': len(listed), 'removed': int((~listed.isin(keep)).sum()),
                     'share_removed': float((~listed.isin(keep)).mean()),
                     'graph_parts': components(ci, ki)[0] if len(gg) else 0,
                     'cities_lost': int(A_SCORE[g].notna().sum() - gg.city.nunique())})
    return pd.DataFrame(rows).set_index('domain')


SCREEN_STATUS, SCREEN_KEEP, SCREEN_HASHES = 'absent', None, {}
if SCREEN_CODES.exists():
    codes = pd.read_csv(SCREEN_CODES, dtype=str).fillna('')
    for col in ['coder_A', 'coder_B', 'final_decision']:
        codes[col] = codes[col].str.strip().str.lower()
    problems = []
    if sorted(codes.keyword) != sorted(dictionary.keyword):
        problems.append('keyword list differs from the 145-keyword dictionary')
    for col in ['coder_A', 'coder_B', 'final_decision']:
        bad = ~codes[col].isin(['keep', 'drop'])
        if bad.any():
            problems.append(f'{col}: {int(bad.sum())} rows are not keep/drop')
    if not SCREEN_RULES.exists():
        problems.append('rules file missing')
    if not problems:
        keep = set(codes.loc[codes.final_decision == 'keep', 'keyword'])
        rep = screen_report(keep)
        print(f"Cohen's kappa between coders: {cohen_kappa(codes.coder_A, codes.coder_B):.3f}")
        display(rep)
        if (rep.share_removed > SCREEN_MAX_DROP + 1e-12).any():
            problems.append(f'more than {SCREEN_MAX_DROP:.0%} removed in: {rep.index[rep.share_removed > SCREEN_MAX_DROP].tolist()}')
    if problems:
        SCREEN_STATUS = 'incomplete'
        print('keyword screen not usable yet:', problems)
    elif not SCREEN_FROZEN:
        SCREEN_STATUS = 'complete, not frozen'
        print('keyword screen complete; set SCREEN_FROZEN = True when the rules and codings are final')
    else:
        SCREEN_STATUS, SCREEN_KEEP = 'frozen', keep
        SCREEN_HASHES = {p.name: sha256(p) for p in [SCREEN_CODES, SCREEN_RULES]}
        print('keyword screen frozen:', SCREEN_HASHES)
print('screen status:', SCREEN_STATUS)

if SCREEN_KEEP is None and MODE == 'dry_run':
    srng = np.random.default_rng(SEED + 2)
    drop = []
    for g in DOMAINS:
        obs_kw = sorted(cells[cells.domain == g].keyword.unique())
        drop += list(srng.choice(obs_kw, size=int(0.1 * len(obs_kw)), replace=False))
    SCREEN_KEEP = set(dictionary.keyword) - set(drop)
    SCREEN_STATUS = 'dry run: random placeholder screen'
    print('dry run uses a random placeholder screen, dropping:', sorted(drop))
    display(screen_report(SCREEN_KEEP))

Cohen's kappa between coders: 0.754


,listed,removed,share_removed,graph_parts,cities_lost
domain,,,,,
Public Transportation,22,4,0.181818,1,0
Driving and Traffic,11,1,0.090909,1,0
Economy and Income,13,1,0.076923,1,0
Education,5,0,0.000000,1,0
Culture and Art,7,0,0.000000,1,0
Race/Immigration Discourse,13,2,0.153846,1,5
Recreation,14,1,0.071429,1,0
Social Problem,20,3,0.150000,1,0
Governing,15,4,0.266667,1,0


keyword screen frozen: {'keyword_screen_codes.csv': 'da1cf7868a89b5e91b16dbd2dc68e4c04d7e17ffbd6da477f7156f0f85bcb40e', 'keyword_screen_rules.md': 'ca28d6120a2a7b75c3028c494a9e8b2177b7010c4e20c92aa887b0880774674e'}
screen status: frozen


## 7. Association, Decomposition, and Robustness Analysis (Algorithm 2; Algorithm 3, Lines 1–8)

**Execution Modes**
- `MODE = 'confirmatory'`: runs on unshuffled urban outcomes and controls; writes all 17 result tables and `run_log.json` to `results/www_short/confirmatory_<timestamp>/`.
- `MODE = 'dry_run'`: jointly permutes outcomes, controls, and urban indicators across cities into `results/www_short/dry_run/`.

**Shared Inference Rule**
All baseline control sets include $\log s_i$; models use HC3 robust standard errors, residual-df $t$-tests, and Benjamini–Hochberg correction within each named family.

In [34]:
RUN_STAMP = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
if MODE == 'confirmatory':
    if SCREEN_STATUS == 'frozen':
        RUN_DIR = OUT / ('confirmatory' if not (OUT / 'confirmatory').exists() else f'confirmatory_{RUN_STAMP}')
        print('REAL DATA: full confirmatory run with the frozen keyword screen.')
    elif MAIN_RUN_BEFORE_SCREEN:
        RUN_DIR = OUT / f'confirmatory_main_{RUN_STAMP}'
        print('REAL DATA: main analysis before the keyword screen is frozen; the screened rerun (7.8) waits.')
    else:
        raise RuntimeError('Confirmatory run blocked: freeze the keyword screen first (section 6), '
                           'or set MAIN_RUN_BEFORE_SCREEN = True to run only the main analysis now.')
    earlier = sorted(p.name for p in OUT.glob('confirmatory*'))
    if earlier:
        print('Earlier confirmatory runs exist and are kept:', earlier)
    ADMIN, Y = aligned.copy(), OUT_Y.copy()
elif MODE == 'dry_run':
    RUN_DIR = OUT / 'dry_run'
    perm = np.random.default_rng(SEED + 1).permutation(len(CITIES))
    ADMIN = aligned.iloc[perm].copy()
    ADMIN.index = CITY_IDX
    Y = OUT_Y.iloc[perm].copy()
    Y.index = CITY_IDX
    print('DRY RUN: the census block (outcomes, controls, indicators) is shuffled across cities. Nothing below is a result.')
else:
    raise ValueError(MODE)
RUN_DIR.mkdir(parents=True, exist_ok=True)
LOG_S = SIZE.log_size.rename('log_s')
LOG_POP = np.log(ADMIN[' pop_2010']).rename('log_pop')
TRANSPORT = ['Public Transportation', 'Driving and Traffic']
HIGHLIGHT = ['Public Transportation', 'Driving and Traffic', 'Social Problem']
print('outputs:', RUN_DIR.relative_to(PROJECT))

REAL DATA: full confirmatory run with the frozen keyword screen.
outputs: results/www_short/confirmatory


### 7.1 Score-Side 12-Domain Decomposition (Algorithm 2, Line 4)

Simultaneous 12-domain OLS models across all 6 outcomes under four predictor matrices:
- `a_zero_fill (original)`: archived score, missing cells set to 0
- `a_mean_fill`: archived score, missing cells set to domain mean
- `tone_mean_fill`: city tone $\hat\alpha_{ig}$, missing cells set to domain mean
- `composition_mean_fill`: keyword composition $c_{ig}$, missing cells set to domain mean

In [35]:
def fill_mean(frame):
    return frame.fillna(frame.mean())


def split_models(sig, Yt, label, designs=None):
    designs = designs or {'a_zero_fill (original)': sig['a'].fillna(0.0), 'a_mean_fill': fill_mean(sig['a']),
                          'tone_mean_fill': fill_mean(sig['tone']), 'composition_mean_fill': fill_mean(sig['comp'])}
    rows = []
    for dname, X in designs.items():
        for oc in OUTCOMES:
            model, robust, _, _ = fit_model(X, Yt[oc])
            rows += coefficient_rows(model, robust, label, oc, f'{label}|{dname}|{oc}', design=dname)
    return add_fdr(pd.DataFrame(rows), ['family'])


SPLIT = split_models(SIG, Y, 'score_side_split')
view = SPLIT[SPLIT.outcome.eq('archived_composite') & SPLIT.term.isin(HIGHLIGHT)]
display(view.pivot(index='term', columns='design', values='beta').loc[HIGHLIGHT].round(3))
display(view.pivot(index='term', columns='design', values='hc3_q_bh').loc[HIGHLIGHT].round(3))

design,a_mean_fill,a_zero_fill (original),composition_mean_fill,tone_mean_fill
term,,,,
Public Transportation,-0.335,-0.324,-0.311,0.063
Driving and Traffic,-0.306,-0.374,-0.149,0.030
Social Problem,-0.432,-0.421,0.019,-0.228


design,a_mean_fill,a_zero_fill (original),composition_mean_fill,tone_mean_fill
term,,,,
Public Transportation,0.204,0.209,0.941,0.944
Driving and Traffic,0.204,0.209,0.941,0.944
Social Problem,0.108,0.209,0.941,0.944


### 7.2 Two-Sided Decomposition of Composite $C$

Align outcome-side demographic components ($b_C = b_U + b_V + b_W$) alongside score-side decompositions (`a_zero_fill`, `a_mean_fill`, `composition_mean_fill`, `tone_mean_fill`).

In [36]:
uvw = outcome_side_split(SIG['a'].fillna(0.0), Y.archived_composite, Y.combined_group_share, Y.normalized_composite)
score_side = SPLIT[SPLIT.outcome.eq('archived_composite') & SPLIT.term.ne('const')].pivot(index='term', columns='design', values='beta')
SIDE = pd.concat([uvw.drop(index='const').add_prefix('outcome_side_'), score_side.add_prefix('score_side_')], axis=1).loc[DOMAINS]
check('both sides start from the same coefficient',
      np.allclose(SIDE.outcome_side_C, SIDE['score_side_a_zero_fill (original)'], atol=1e-10))
display(SIDE.round(3))

PASS  both sides start from the same coefficient


,outcome_side_C,outcome_side_U_share,outcome_side_V_normalized,outcome_side_W_interaction,score_side_a_mean_fill,score_side_a_zero_fill (original),score_side_composition_mean_fill,score_side_tone_mean_fill
Public Transportation,-0.324,-0.306,0.008,-0.026,-0.335,-0.324,-0.311,0.063
Driving and Traffic,-0.374,-0.404,0.036,-0.006,-0.306,-0.374,-0.149,0.030
Economy and Income,-0.383,-0.461,0.044,0.034,-0.528,-0.383,-0.060,-0.148
Education,0.054,0.182,-0.127,-0.001,0.004,0.054,-0.192,-0.015
Culture and Art,0.107,0.052,0.073,-0.019,0.019,0.107,-0.043,0.077
Race/Immigration Discourse,0.119,0.153,-0.043,0.009,0.049,0.119,0.085,-0.086
Recreation,0.147,0.176,-0.034,0.005,0.140,0.147,-0.033,-0.053
Social Problem,-0.421,-0.285,-0.174,0.038,-0.432,-0.421,0.019,-0.228
Governing,-0.374,-0.365,-0.021,0.011,-0.306,-0.374,0.206,-0.093
Policing,0.087,0.098,-0.022,0.010,-0.033,0.087,-0.042,0.390


### 7.3 Single-Domain Model Grid (Algorithm 2, Lines 5–6; 864 Models)

12 domains $\times$ 6 outcomes $\times$ 3 control sets (+ $\log s_i$) $\times$ 4 signals (listwise score, zero-filled score, tone, coverage), BH-adjusted within each 12-domain family.

In [37]:
def grid_models(sig, Yt, admin, label, domains=DOMAINS, rules=SIGNAL_RULES, outcomes=OUTCOMES):
    rows = []
    for rule in rules:
        for tag, ctrls in CTRL_SETS.items():
            for oc in outcomes:
                fam = f'{label}|{rule}|{tag}+log_s|{oc}'
                for g in domains:
                    x = signal(sig, rule, g).rename(g)
                    ok = x.notna()
                    X = pd.concat([x, admin[ctrls], LOG_S], axis=1).loc[ok]
                    model, robust, _, _ = fit_model(X, Yt.loc[ok, oc])
                    rows += coefficient_rows(model, robust, label, oc, fam, focal_domain=g, signal=rule, controls=f'{tag}+log_s')
    out = pd.DataFrame(rows)
    return add_fdr(out, ['family'], out.term.eq(out.focal_domain))


t0 = time.time()
GRID = grid_models(SIG, Y, ADMIN, 'single_domain_grid')
GRID_FOCAL = GRID[GRID.term.eq(GRID.focal_domain)]
check('864 single-domain models', len(GRID_FOCAL) == 864, f'{time.time() - t0:.0f} s')
v = GRID_FOCAL[GRID_FOCAL.outcome.eq('archived_composite') & GRID_FOCAL.focal_domain.isin(TRANSPORT)]
display(v.pivot_table(index=['focal_domain', 'signal'], columns='controls', values='beta').round(3))
print('focal coefficients with HC3 q < .05:', int((GRID_FOCAL.hc3_q_bh < .05).sum()), 'of 864')

PASS  864 single-domain models   [2 s]


controls                           density_income+log_s  density_income_shares+log_s  none+log_s
focal_domain          signal                                                                    
Driving and Traffic   a_available                -0.036                        0.037      -0.153
                      a_zero_fill                -0.036                        0.037      -0.153
                      coverage_n                  0.166                        0.156      -0.045
                      tone                       -0.070                        0.019      -0.135
Public Transportation a_available                -0.127                       -0.042      -0.179
                      a_zero_fill                -0.122                       -0.014      -0.209
                      coverage_n                  0.405                        0.064       0.241
                      tone                        0.246                       -0.029       0.020

focal coefficients with HC3 q < .05: 9 of 864


### 7.4 RQ2: Partial Correlations with Urban Indicators (Algorithm 2, Lines 7–9)

Pearson and Spearman partial correlations controlling for $\log s_i$ and $\log(\text{population})$, BH-adjusted across all 132 pairs per signal and method, plus 16 exploratory public-transport checks (`Transit Score` and `commute_%public transit`) and breakdown of the 4 unadjusted baseline hits.

In [38]:
def partial_corr(x, y, Z, rank=False):
    M = pd.concat([x.rename('x'), y.rename('y'), Z], axis=1).dropna()
    if rank:
        M = M.rank()
    Zm = np.column_stack([np.ones(len(M)), M.iloc[:, 2:].to_numpy(float)])
    rx = M.x - Zm @ np.linalg.lstsq(Zm, M.x, rcond=None)[0]
    ry = M.y - Zm @ np.linalg.lstsq(Zm, M.y, rcond=None)[0]
    r = np.corrcoef(rx, ry)[0, 1]
    df = len(M) - 2 - (Zm.shape[1] - 1)
    return r, 2 * stats.t.sf(abs(r * np.sqrt(df / (1 - r ** 2))), df), len(M)


def rq2_partials(sig, admin, label, domains=DOMAINS, indicators=TABLE3_PREDICTORS):
    Z = pd.concat([LOG_S, np.log(admin[' pop_2010']).rename('log_pop')], axis=1)
    rows = []
    for sname in ['tone', 'n']:
        for method in ['pearson', 'spearman']:
            for g in domains:
                for ind in indicators:
                    r, p, n = partial_corr(sig[sname][g], admin[ind], Z, rank=(method == 'spearman'))
                    rows.append({'signal': sname, 'method': method, 'domain': g, 'indicator': ind, 'n': n, 'partial_r': r, 'p': p})
    out = pd.DataFrame(rows)
    out['family'] = label + '|' + out.signal + '|' + out.method + f'|{len(domains) * len(indicators)}'
    out['q_bh'] = out.groupby('family').p.transform(lambda p: multipletests(p, method='fdr_bh')[1])
    return out


RQ2 = rq2_partials(SIG, ADMIN, 'rq2_partial')
check('4 families of 132 partial correlations', RQ2.groupby('family').size().eq(132).all() and RQ2.family.nunique() == 4)
print(RQ2.assign(passed=RQ2.q_bh < .05).groupby(['signal', 'method']).passed.sum().rename('q < .05').to_frame().T)

RQ2_EXTRA = rq2_partials(SIG, ADMIN, 'rq2_exploratory_transport', domains=TRANSPORT,
                         indicators=['Transit Score', 'commute_%public transit'])
RQ2_EXTRA['family'] = 'rq2_exploratory_transport|16'
RQ2_EXTRA['q_bh'] = multipletests(RQ2_EXTRA.p, method='fdr_bh')[1]
display(RQ2_EXTRA.round(3))

ORIGINAL_PAIRS = [('Neighborhood', 'commute_%drive', 'pearson'), ('Recreation', 'Driving Score', 'pearson'),
                  ('Economy and Income', 'commute_%drive', 'spearman'), ('Education', 'Black_White income inequality', 'spearman')]
rows = []
for g, ind, method in ORIGINAL_PAIRS:
    pair = pd.concat([A_SCORE[g], ADMIN[ind]], axis=1).dropna()
    r_a = (stats.pearsonr if method == 'pearson' else stats.spearmanr)(pair.iloc[:, 0], pair.iloc[:, 1]).statistic
    sel = RQ2[(RQ2.domain == g) & (RQ2.indicator == ind) & (RQ2.method == method)].set_index('signal')
    rows.append({'domain': g, 'indicator': ind, 'method': method, 'r_score_raw': r_a,
                 'partial_r_coverage': sel.loc['n', 'partial_r'], 'q_coverage': sel.loc['n', 'q_bh'],
                 'partial_r_tone': sel.loc['tone', 'partial_r'], 'q_tone': sel.loc['tone', 'q_bh']})
FOUR_PAIRS = pd.DataFrame(rows)
display(FOUR_PAIRS.round(3))

PASS  4 families of 132 partial correlations
signal        n             tone         
method  pearson spearman pearson spearman
q < .05       2        2       0        2


,signal,method,domain,indicator,n,partial_r,p,family,q_bh
0,tone,pearson,Public Transportation,Transit Score,37,-0.175,0.315,rq2_exploratory_transport|16,0.447
1,tone,pearson,Public Transportation,commute_%public transit,37,-0.168,0.335,rq2_exploratory_transport|16,0.447
2,tone,pearson,Driving and Traffic,Transit Score,38,-0.284,0.094,rq2_exploratory_transport|16,0.197
3,tone,pearson,Driving and Traffic,commute_%public transit,38,-0.280,0.098,rq2_exploratory_transport|16,0.197
4,tone,spearman,Public Transportation,Transit Score,37,-0.187,0.281,rq2_exploratory_transport|16,0.447
5,tone,spearman,Public Transportation,commute_%public transit,37,-0.383,0.023,rq2_exploratory_transport|16,0.074
6,tone,spearman,Driving and Traffic,Transit Score,38,-0.308,0.068,rq2_exploratory_transport|16,0.182
7,tone,spearman,Driving and Traffic,commute_%public transit,38,-0.251,0.139,rq2_exploratory_transport|16,0.248
8,n,pearson,Public Transportation,Transit Score,38,0.048,0.780,rq2_exploratory_transport|16,0.780
9,n,pearson,Public Transportation,commute_%public transit,38,0.117,0.498,rq2_exploratory_transport|16,0.613


,domain,indicator,method,r_score_raw,partial_r_coverage,q_coverage,partial_r_tone,q_tone
0,Neighborhood,commute_%drive,pearson,0.551,-0.039,0.879,0.331,0.416
1,Recreation,Driving Score,pearson,0.540,-0.073,0.801,0.040,0.966
2,Economy and Income,commute_%drive,spearman,0.606,-0.158,0.605,0.388,0.259
3,Education,Black_White income inequality,spearman,0.572,0.049,0.903,0.595,0.017


### 7.5 End-to-End City Bootstrap (Algorithm 3, Lines 1–5)

Resample 38 cities with replacement ($B = 2000$), re-estimate Algorithm 1 within the largest connected component of each resample, and record percentile intervals for $\Delta = b(a) - b(\text{tone})$ in both 12-domain and single-domain transport models.

In [39]:
def signals_from_draw(draw, domains=DOMAINS):
    """Algorithm 1 on a resample; resampled city j is its own city even if drawn twice."""
    nC = len(draw)
    out = {k: np.full((nC, len(domains)), np.nan) for k in ['a', 'tone', 'comp']}
    split = {}
    for j, g in enumerate(domains):
        ci, ki, y = resample_domain(g, draw)
        f, split[g] = fe_connected(ci, ki, y)
        out['tone'][f['uc'], j] = f['tone']
        out['comp'][f['uc'], j] = f['comp']
        cnt = np.bincount(ci, minlength=nC)
        with np.errstate(invalid='ignore', divide='ignore'):
            out['a'][:, j] = np.where(cnt > 0, np.bincount(ci, weights=y, minlength=nC) / np.maximum(cnt, 1), np.nan)
    return out, split


def col_mean_fill(M):
    M = M.copy()
    mu = np.nanmean(M, axis=0)
    r, c = np.where(np.isnan(M))
    M[r, c] = mu[c]
    return M


TJ = [DOMAINS.index(g) for g in TRANSPORT]
YC = Y.archived_composite.to_numpy(float)
ZC = {tag: np.column_stack([ADMIN[ctrls].to_numpy(float), LOG_S.to_numpy(float)]) if ctrls else LOG_S.to_numpy(float)[:, None]
      for tag, ctrls in CTRL_SETS.items()}


def boot_stats(draw):
    s, split = signals_from_draw(draw)
    yb = YC[draw]
    ba, bt = std_coefs(yb, col_mean_fill(s['a'])), std_coefs(yb, col_mean_fill(s['tone']))
    out = {}
    for g, j in zip(TRANSPORT, TJ):
        out[f'12-domain | {g}'] = ba[j] - bt[j]
        ok = ~np.isnan(s['tone'][:, j])
        for tag in CTRL_SETS:
            Zb = ZC[tag][draw][ok]
            out[f'single-domain | {g} | {tag}+log_s'] = (std_coefs(yb[ok], np.column_stack([s['a'][ok, j], Zb]))[0]
                                                         - std_coefs(yb[ok], np.column_stack([s['tone'][ok, j], Zb]))[0])
    return out, split


point, _ = boot_stats(np.arange(len(CITIES)))
t0 = time.time()
draws, splits = [], {g: 0 for g in DOMAINS}
boot_rng = np.random.default_rng(SEED + 3)
for b in range(B_BOOT):
    st, sp = boot_stats(boot_rng.integers(0, len(CITIES), len(CITIES)))
    draws.append(st)
    for g in DOMAINS:
        splits[g] += sp[g]
D = pd.DataFrame(draws)
BOOT = pd.DataFrame({'delta_point': pd.Series(point), 'q2.5': D.quantile(.025), 'q97.5': D.quantile(.975),
                     'valid_resamples': D.notna().sum()})
print(f'{B_BOOT} resamples, {time.time() - t0:.0f} s | resamples with a split graph:',
      {g[:14]: v for g, v in splits.items() if v})
display(BOOT.round(3))

2000 resamples, 9 s | resamples with a split graph: {'Race/Immigrati': 171}


,delta_point,q2.5,q97.5,valid_resamples
12-domain | Public Transportation,-0.398,-0.935,0.128,2000
single-domain | Public Transportation | none+log_s,-0.198,-0.563,0.239,2000
single-domain | Public Transportation | density_income+log_s,-0.373,-0.672,-0.002,2000
single-domain | Public Transportation | density_income_shares+log_s,-0.013,-0.115,0.095,2000
12-domain | Driving and Traffic,-0.336,-0.830,0.075,2000
single-domain | Driving and Traffic | none+log_s,-0.019,-0.334,0.252,2000
single-domain | Driving and Traffic | density_income+log_s,0.034,-0.218,0.268,2000
single-domain | Driving and Traffic | density_income_shares+log_s,0.018,-0.062,0.094,2000


### 7.6 Leave-One-City-Out, Leave-One-Keyword-Out, and Rank Sensitivity (Algorithm 3, Line 6)

Check sign and magnitude stability under single-city deletion (fixed full-sample scaling), single-keyword deletion (re-estimating Algorithm 1), and rank-transformed single-domain regressions.

In [40]:
SPLIT_DESIGNS = {'a_mean_fill': lambda s: fill_mean(s['a']), 'tone_mean_fill': lambda s: fill_mean(s['tone']),
                 'composition_mean_fill': lambda s: fill_mean(s['comp'])}
yC = Y.archived_composite

# leave one city out, full-sample scaling held fixed
rows = []
for dname, make in SPLIT_DESIGNS.items():
    X = sm.add_constant(zscore(make(SIG)))
    yz = zscore(yC)
    full = sm.OLS(yz, X).fit().params
    for i, city in enumerate(CITY_IDX):
        keep = np.arange(len(CITY_IDX)) != i
        pr = sm.OLS(yz[keep], X[keep]).fit().params
        for g in HIGHLIGHT:
            rows.append({'model': f'12-domain | {dname}', 'term': g, 'omitted': city, 'beta_full': full[g], 'beta_loo': pr[g]})
for g in TRANSPORT:
    for rule in ['a_available', 'tone', 'coverage_n']:
        for tag, ctrls in CTRL_SETS.items():
            x = signal(SIG, rule, g).rename(g)
            ok = x.notna()
            Xf = pd.concat([x, ADMIN[ctrls], LOG_S], axis=1).loc[ok]
            X = sm.add_constant(zscore(Xf))
            yz = zscore(yC[ok])
            full = sm.OLS(yz, X).fit().params[g]
            for city in X.index:
                pr = sm.OLS(yz.drop(city), X.drop(city)).fit().params[g]
                rows.append({'model': f'single-domain | {rule} | {tag}+log_s', 'term': g, 'omitted': city, 'beta_full': full, 'beta_loo': pr})
LOO_CITY = pd.DataFrame(rows)
LOO_CITY_SUM = LOO_CITY.groupby(['model', 'term'], sort=False).agg(
    beta_full=('beta_full', 'first'), beta_min=('beta_loo', 'min'), beta_max=('beta_loo', 'max'),
    sign_consistent=('beta_loo', lambda s: int((np.sign(s) == np.sign(LOO_CITY.loc[s.index, 'beta_full'])).sum())),
    omissions=('beta_loo', 'size')).reset_index()
display(LOO_CITY_SUM[LOO_CITY_SUM.model.str.startswith('12-domain')].round(3))

,model,term,beta_full,beta_min,beta_max,sign_consistent,omissions
0,12-domain | a_mean_fill,Public Transportation,-0.335,-0.406,-0.271,38,38
1,12-domain | a_mean_fill,Driving and Traffic,-0.306,-0.391,-0.262,38,38
2,12-domain | a_mean_fill,Social Problem,-0.432,-0.491,-0.390,38,38
3,12-domain | tone_mean_fill,Public Transportation,0.063,-0.044,0.171,36,38
4,12-domain | tone_mean_fill,Driving and Traffic,0.030,-0.064,0.100,35,38
5,12-domain | tone_mean_fill,Social Problem,-0.228,-0.338,-0.181,38,38
6,12-domain | composition_mean_fill,Public Transportation,-0.311,-0.462,-0.207,38,38
7,12-domain | composition_mean_fill,Driving and Traffic,-0.149,-0.684,-0.042,38,38
8,12-domain | composition_mean_fill,Social Problem,0.019,-0.112,0.084,27,38


In [41]:
# leave one keyword out: rerun Algorithm 1 for that keyword's domain, then refit
def coefs_from_signals(sig):
    out = {}
    for dname, make in SPLIT_DESIGNS.items():
        b = std_coefs(yC.to_numpy(float), make(sig).to_numpy(float))
        for g in HIGHLIGHT:
            out[f'12-domain | {dname} | {g}'] = b[DOMAINS.index(g)]
    for g in TRANSPORT:
        for rule in ['a_available', 'tone', 'coverage_n']:
            for tag, ctrls in CTRL_SETS.items():
                x = signal(sig, rule, g)
                ok = x.notna().to_numpy()
                Zt = np.column_stack([ADMIN[ctrls].to_numpy(float), LOG_S.to_numpy(float)]) if ctrls else LOG_S.to_numpy(float)[:, None]
                out[f'single-domain | {rule} | {tag}+log_s | {g}'] = std_coefs(yC.to_numpy(float)[ok], np.column_stack([x.to_numpy(float)[ok], Zt[ok]]))[0]
    return out


base = coefs_from_signals(SIG)
rows = []
t0 = time.time()
for kw in sorted(cells.keyword.unique()):
    g = cells.loc[cells.keyword == kw, 'domain'].iloc[0]
    sub, _ = measure(cells[cells.keyword != kw], domains=[g])
    sig_k = {k: SIG[k].copy() for k in SIG}
    for k in sig_k:
        sig_k[k][g] = sub[k][g]
    rows.append({'dropped_keyword': kw, 'domain': g, **coefs_from_signals(sig_k)})
LOO_KW = pd.DataFrame(rows)
stats_cols = [c for c in LOO_KW.columns if '|' in c]
LOO_KW_SUM = pd.DataFrame({'beta_all_keywords': pd.Series(base), 'beta_min': LOO_KW[stats_cols].min(), 'beta_max': LOO_KW[stats_cols].max(),
                           'sign_flips': (np.sign(LOO_KW[stats_cols]) != np.sign(pd.Series(base)[stats_cols])).sum(),
                           'most_influential_keyword': LOO_KW[stats_cols].sub(pd.Series(base)[stats_cols]).abs().idxmax().map(LOO_KW.dropped_keyword)})
print(f'{len(LOO_KW)} keywords dropped one at a time, {time.time() - t0:.0f} s')
display(LOO_KW_SUM[LOO_KW_SUM.index.str.startswith('12-domain')].round(3))

143 keywords dropped one at a time, 1 s


,beta_all_keywords,beta_min,beta_max,sign_flips,most_influential_keyword
12-domain | a_mean_fill | Public Transportation,-0.335,-0.402,-0.213,0,transit
12-domain | a_mean_fill | Driving and Traffic,-0.306,-0.369,-0.175,0,prices
12-domain | a_mean_fill | Social Problem,-0.432,-0.498,-0.362,0,law
12-domain | tone_mean_fill | Public Transportation,0.063,-0.056,0.168,3,transportation
12-domain | tone_mean_fill | Driving and Traffic,0.030,-0.018,0.079,3,police
12-domain | tone_mean_fill | Social Problem,-0.228,-0.281,-0.052,0,crime
12-domain | composition_mean_fill | Public Transportation,-0.311,-0.555,0.032,1,transit
12-domain | composition_mean_fill | Driving and Traffic,-0.149,-0.351,0.035,2,direction
12-domain | composition_mean_fill | Social Problem,0.019,-0.239,0.134,16,homeless


In [42]:
# rank (Spearman-type) versions of the transport single-domain models, outcome C
rows = []
for g in TRANSPORT:
    for rule in ['a_available', 'tone', 'coverage_n']:
        for tag, ctrls in CTRL_SETS.items():
            x = signal(SIG, rule, g).rename(g)
            ok = x.notna()
            Xr = pd.concat([x, ADMIN[ctrls], LOG_S], axis=1).loc[ok].rank()
            model, robust, _, _ = fit_model(Xr, yC[ok].rank())
            rows += [r for r in coefficient_rows(model, robust, 'rank_version', 'archived_composite', f'rank|{rule}|{tag}+log_s',
                                                 focal_domain=g, signal=rule, controls=f'{tag}+log_s') if r['term'] == g]
RANK = pd.DataFrame(rows)
display(RANK.pivot_table(index=['term', 'signal'], columns='controls', values='beta').round(3))

controls                           density_income+log_s  density_income_shares+log_s  none+log_s
term                  signal                                                                    
Driving and Traffic   a_available                -0.192                        0.057      -0.288
                      coverage_n                  0.098                       -0.025      -0.011
                      tone                       -0.004                        0.214      -0.018
Public Transportation a_available                -0.209                       -0.057      -0.135
                      coverage_n                  0.479                        0.076       0.349
                      tone                        0.182                       -0.000       0.092

### 7.7 Random-Effects BLUP Tone Version (Algorithm 3, Line 7)

Replace fixed-effects tone with REML BLUP tone in the 8 domains passing $p_{\text{boundary}} < 0.20$ and re-run both the 12-domain and single-domain tone models.

In [43]:
SIG_RE = {k: v.copy() for k, v in SIG.items()}
RE_DOMAINS = [g for g in DOMAINS if RE[g]['eligible']]
for g in RE_DOMAINS:
    SIG_RE['tone'][g] = RE[g]['u']
SPLIT_RE = split_models(SIG_RE, Y, 'score_side_split_RE_tone', designs={'tone_mean_fill (BLUP where eligible)': fill_mean(SIG_RE['tone'])})
GRID_RE = grid_models(SIG_RE, Y, ADMIN, 'single_domain_grid_RE_tone', domains=RE_DOMAINS, rules=['tone'])
GRID_RE_FOCAL = GRID_RE[GRID_RE.term.eq(GRID_RE.focal_domain)]
cmp = pd.DataFrame({'fixed-effects tone': SPLIT[(SPLIT.design == 'tone_mean_fill') & SPLIT.outcome.eq('archived_composite')].set_index('term').beta,
                    'BLUP tone': SPLIT_RE[SPLIT_RE.outcome.eq('archived_composite')].set_index('term').beta}).loc[DOMAINS]
display(cmp.round(3))

,fixed-effects tone,BLUP tone
term,,
Public Transportation,0.063,-0.045
Driving and Traffic,0.030,0.066
Economy and Income,-0.148,-0.158
Education,-0.015,0.054
Culture and Art,0.077,0.104
Race/Immigration Discourse,-0.086,-0.107
Recreation,-0.053,-0.057
Social Problem,-0.228,-0.200
Governing,-0.093,-0.109


### 7.8 Screened-Vocabulary Version (Algorithm 3, Line 8)

Restrict to the 125 keywords retained after double coding and re-run Algorithm 1, score-side 12-domain decompositions, the single-domain grid, and RQ2 partial correlations.

In [44]:
if SCREEN_KEEP is not None:
    cells_s = cells[cells.keyword.isin(SCREEN_KEEP)]
    SIG_S, FITS_S = measure(cells_s)
    print('graphs split after screening:', [g for g in DOMAINS if FITS_S[g]['parts'] > 1])
    SPLIT_S = split_models(SIG_S, Y, 'screened_score_side_split')
    GRID_S = grid_models(SIG_S, Y, ADMIN, 'screened_single_domain_grid')
    RQ2_S = rq2_partials(SIG_S, ADMIN, 'screened_rq2_partial')
    both = pd.concat({'all keywords': SPLIT.set_index(['design', 'outcome', 'term']).beta,
                      'screened': SPLIT_S.set_index(['design', 'outcome', 'term']).beta}, axis=1)
    bc = both.xs('archived_composite', level='outcome')
    display(bc[bc.index.get_level_values('term').isin(HIGHLIGHT)].round(3))
else:
    SPLIT_S = GRID_S = RQ2_S = None
    print('no frozen keyword screen yet; screened rerun skipped (run it once after freezing)')

graphs split after screening: []


all keywords  screened
design                 term                                         
a_zero_fill (original) Public Transportation        -0.324    -0.263
                       Driving and Traffic          -0.374    -0.352
                       Social Problem               -0.421    -0.344
a_mean_fill            Public Transportation        -0.335    -0.326
                       Driving and Traffic          -0.306    -0.375
                       Social Problem               -0.432    -0.347
tone_mean_fill         Public Transportation         0.063     0.054
                       Driving and Traffic           0.030     0.059
                       Social Problem               -0.228    -0.252
composition_mean_fill  Public Transportation        -0.311    -0.249
                       Driving and Traffic          -0.149    -0.349
                       Social Problem                0.019    -0.053

### 7.9 Save Outputs and Provenance Log

In [45]:
tables = {'score_side_split.csv': SPLIT, 'two_sided_decomposition_C.csv': SIDE.reset_index(names='domain'),
          'single_domain_grid_864.csv': GRID, 'rq2_partial_correlations.csv': RQ2, 'rq2_exploratory_transport.csv': RQ2_EXTRA,
          'rq2_original_four_pairs.csv': FOUR_PAIRS, 'bootstrap_delta.csv': BOOT.reset_index(names='statistic'),
          'bootstrap_delta_draws.csv': D, 'leave_one_city_out.csv': LOO_CITY_SUM, 'leave_one_keyword_out.csv': LOO_KW,
          'leave_one_keyword_out_summary.csv': LOO_KW_SUM.reset_index(names='statistic'), 'rank_versions.csv': RANK,
          'score_side_split_RE_tone.csv': SPLIT_RE, 'single_domain_grid_RE_tone.csv': GRID_RE}
if SPLIT_S is not None:
    tables.update({'screened_score_side_split.csv': SPLIT_S, 'screened_single_domain_grid.csv': GRID_S,
                   'screened_rq2_partial_correlations.csv': RQ2_S})
for name, frame in tables.items():
    save(frame, RUN_DIR, name)
NOTEBOOK_FILE = PROJECT / 'code' / 'www_short_analysis.ipynb'
code_cells = [c for c in json.loads(NOTEBOOK_FILE.read_text(encoding='utf-8'))['cells'] if c['cell_type'] == 'code']
CODE_SHA = hashlib.sha256('\n'.join(''.join(c['source']) for c in code_cells[1:]).encode()).hexdigest()
log = {'mode': MODE, 'run_stamp': RUN_STAMP, 'main_run_before_screen': bool(MODE == 'confirmatory' and SCREEN_STATUS != 'frozen'),
       'analysis_code_sha256 (all code cells except settings)': CODE_SHA, 'note': 'DRY RUN: census block shuffled across cities; not results' if MODE == 'dry_run' else 'confirmatory',
       'input_sha256': INPUT_HASHES, 'screen_status': SCREEN_STATUS, 'screen_sha256': SCREEN_HASHES,
       'settings': {'seed': SEED, 'quick': QUICK, 'B_BOOT': B_BOOT, 'R_NULL': R_NULL, 'retention_threshold': RETENTION_THRESHOLD,
                    'theta': round(float(THETA), 4), 'screen_max_drop': SCREEN_MAX_DROP, 're_lrt_alpha': RE_LRT_ALPHA,
                    're_tone_domains': RE_DOMAINS, 'control_sets': {k: v + ['log s'] for k, v in CTRL_SETS.items()}},
       'software': {'python': platform.python_version(), 'numpy': np.__version__, 'pandas': pd.__version__,
                    'scipy': scipy.__version__, 'statsmodels': statsmodels.__version__},
       'checks_passed': sum(c['passed'] for c in CHECKS), 'checks': CHECKS,
       'runtime_seconds': round(time.time() - T_START, 1)}
(RUN_DIR / 'run_log.json').write_text(json.dumps(log, indent=2, ensure_ascii=False, default=str), encoding='utf-8')
print(f'{len(tables)} tables and run_log.json written to {RUN_DIR.relative_to(PROJECT)}')

17 tables and run_log.json written to results/www_short/confirmatory


## 8. Output Inventory

In [46]:
for folder in sorted(p for p in OUT.iterdir() if p.is_dir()):
    files = sorted(f.name for f in folder.iterdir())
    print(f'{folder.relative_to(PROJECT)}  ({len(files)} files)')
    for f in files:
        print('   ', f)
print(f'\n{len(CHECKS)} checks passed; total run time {time.time() - T_START:.0f} s')

results/www_short/confirmatory  (18 files)
    bootstrap_delta.csv
    bootstrap_delta_draws.csv
    leave_one_city_out.csv
    leave_one_keyword_out.csv
    leave_one_keyword_out_summary.csv
    rank_versions.csv
    rq2_exploratory_transport.csv
    rq2_original_four_pairs.csv
    rq2_partial_correlations.csv
    run_log.json
    score_side_split.csv
    score_side_split_RE_tone.csv
    screened_rq2_partial_correlations.csv
    screened_score_side_split.csv
    screened_single_domain_grid.csv
    single_domain_grid_864.csv
    single_domain_grid_RE_tone.csv
    two_sided_decomposition_C.csv
results/www_short/derived  (2 files)
    admin_outcome_side_split.csv
    derived_numbers.txt
results/www_short/diagnostics  (5 files)
    bootstrap_calibration.csv
    coverage_size_and_vif.csv
    keyword_removal_identification.csv
    null_simulation.csv
    permutation_dry_run.csv
results/www_short/keyword_screen  (1 files)
    keyword_screen_TEMPLATE.csv
results/www_short/measurement  (7 file